# <span style="color:aqua">Patient 3455</span>
In this notebook, I will implement all the necessary processing procedures to extract information to analyze the data including visualization to better understand the data. 
Inspired from subjects_loop.ipynb

In [1]:
# TODO:
# - Check again that you are using the correct epochs object (epochs_ref_no_hga for envelopes, epochs_ref for hga, epochs_bands_stats for band-specific statistical analyses) 
#   in each cell. 
# - Maybe add other visualization such as TFRs, ...etc
# - Add more comments to the code to explain what each step is doing, especially for the statistical analyses and the visualization steps.

## <span style="color:aquamarine">Imports</span>


In [2]:
# General Imports
import os
import re
import mne
import neo
import imageio
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Matplotlib imports
import matplotlib 
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# MNE imports
import mne
import mne_bids
from mne import Epochs
from mne_bids import BIDSPath
from mne.datasets import somato
from mne.datasets import fetch_fsaverage
from mne.stats import permutation_cluster_test
from mne.time_frequency import EpochsTFRArray
from mne_bids import (
    find_matching_paths,
    read_raw_bids,
)

# Neuralset imports
import neuralset # TODO: explore the library and understand how to use it for visualization

# Statistical imports
from scipy.signal import hilbert
from scipy.stats import ttest_ind
from scipy.stats import t as t_dist
from mne.stats import fdr_correction
from matplotlib.patches import Patch
from mne.stats import permutation_cluster_test
from scipy.stats import ttest_ind, mannwhitneyu
from mne.stats import permutation_cluster_1samp_test

# Python file functions import 
from epoching import get_epochs_from_raw_filtered
from functions import raw_from_neo, seeg_ch_name_split, find_anodes_cathodes, read_montage, read_events, signed_ttest
from loading_PAT3455 import PAT_3455_loading

# Set up the 3D backend for MNE visualization
matplotlib.use('Agg') # or nbAgg
mne.viz.set_3d_backend('notebook')

EXTRA_COLUMNS_DESCRIPTIONS = {
    'stim_onset_unity': 'Stimulus onset time in seconds since the start of the Unity game',
    'resp_onset_unity': 'Response onset time in seconds since the start of the Unity game',
    'duration': 'Duration between stimulus and response in seconds',
    'condition': 'Condition of the trial (Current, Past, Distant Past, Futur, Distant Futur)',
    'correct': 'Whether the response is correct or not (True/False)',
    'validation': 'whether the response is correct or not(Eprime)',
    'stim': 'Stimulus presented to the subject',
    'resp': 'Response given by the subject',
    'correct_year': 'Year that the subject should have answered',
    'year': 'Year during which the subject was asked to answer',
    'cross_time': 'duration of the fixation cross before trial onset',
}

/home/aboschun/.conda/envs/miplab/lib/python3.14/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Using notebook 3d backend.


## <span style="color:aquamarine"> Parameters </span>

In [3]:
# Choose between TRC data or BIDS data
data_source = "TRC" # "TRC" or "BIDS" or "Check" (for comparing outputs of both)(If check chosen, notebook will be executed with TRC data)

# Electrode Referencing type
reference = "bipolar" # "bipolar" or "bipolar_shaft" or "no_reference" or "average" or "all" (with no video rendering)

# Video rendering
video_rendering = False # can be long, so do not do it if not needed
nb_videos = 3 # number of videos to render (usually low to test the pipeline)

# BIDS data already loaded
loaded = False # Can take up ot 15min

# TODO - solve the patient and subject variables, maybe create a mapping between them
patient = "PAT_3455"
subject = "02"

# Interval around stimulus onset to epoch the data
tmin = -1
tmax = 7 
PAD = 0.5


# Patient directory
if subject == "01":
    experiment_dir = fr"/media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions/PatientHL1996/Experiment_997996.txt"
    trc_dir = fr"/media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions/PatientHL1996/EEG_870458.TRC"
if subject == "02":
    experiment_dir = fr"/media/RCPNAS/sEEG_MARS_Alison/sourcedata/stims/sub-02/P2/Unity_experiment/Experiment_698092.txt"
    trc_dir = r"/media/RCPNAS/sEEG_MARS_Alison/sourcedata/seeg/sub-02/PA_1987/EEG_6865.TRC"


# Check parameters
if data_source not in ["BIDS", "TRC", "Check"]:
    raise ValueError("Data source must be either TRC or BIDS (or Check)")

## <span style="color:aquamarine">Data loading and processing</span>

The next cell will: 
- load the data
- clean the channel names 
- adjust stimulus onset timing
- filter out behaviorally incorrect trials
- crop the signal (and index it correctly)
- integrate annotations of the conditions
- check bad annotation path
- read electrode coordinates and get the montage (and save it) and set it to the raw data
- get volume labels (for electrodes characterization)
- create some vizualisation of the electrodes


In [4]:
# BIDS root
bids_root = Path("/media/RCPNAS/sEEG_MARS_Alison")

# FreeSurfer subjects directory - contains reconstructions for all patients, including PAT3390
fs_subjects_dir = Path("/media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions") 

# Check participants.tsv for mapping
participants_file = bids_root / "participants.tsv"
if participants_file.exists():
    participants = pd.read_csv(participants_file, sep='\t')
    print("Participants mapping:")
    print(participants)

# Verify the PAT3390 subject exists
pat_path = fs_subjects_dir / patient
print(f"PAT path exists: {pat_path.exists()}")
if pat_path.exists():
    print("Found FreeSurfer subject PAT with directories:")
    print([d.name for d in pat_path.iterdir() if d.is_dir()])

# TODO: Now create mapping for all subjects
subject_to_fs = {
    "01": "PAT_3390",
    "02": "PAT_3455"
}

# Process sub-02
session = "retrieval"
task = "mars"
fs_subject = subject_to_fs[subject]
print(f"\nProcessing {subject} → FreeSurfer subject: {fs_subject}")
print(f"FreeSurfer directory: {fs_subjects_dir / fs_subject}")

subjects_dir = bids_root / "sourcedata" / "reconstructions"

# Create BIDS path from bids_root
bids_path = BIDSPath(
    subject=subject,
    session=session,
    task=task,
    datatype="ieeg",
    root=bids_root,
    suffix="ieeg",
    extension=".vhdr"
)

if loaded == False : # Skip the loading step if the data is already loaded and cleaned
    df, raw, montage, annotations = PAT_3455_loading(data_source, bids_root, bids_path, experiment_dir, trc_dir, fs_subjects_dir, subject, patient, session, task)

if loaded == True:
    #TODO: Get the data from the corresponding path
    raw_save_path = f"/home/aboschun/MIPlab-Project/data/raw/raw_sub-{subject}.fif"
    print(f"Loading preprocessed raw from: {raw_save_path}")
    raw = mne.io.read_raw_fif(raw_save_path, preload=True)
    print(f"Loaded raw: {len(raw.ch_names)} channels, {raw.times[-1]:.2f}s")

    # Get the montage
    montage_path = Path.cwd() / f"sub-{subject}_montage.fif"
    if montage_path.exists():
        montage = mne.channels.read_dig_fif(montage_path)
        print(f"Loaded montage from {montage_path}")
    else:
        montage = None
        print("No saved montage found")

    # Get annotations
    annotations_path = f"/home/aboschun/MIPlab-Project/data/annotations/sub-{subject}_annot.fif"
    annotations = mne.read_annotations(annotations_path)
    if annotations is not None and len(annotations) > 0:
        print(f"Loaded {len(annotations)} annotations from file")
        print(f"Example annotation: {annotations[0]}")
    else:
        print("No annotations found in file")
    raw.set_annotations(annotations)
    print(f"Annotations set on raw: {len(raw.annotations)} total")

Participants mapping:
  participant_id  age  sex  hand  weight  height
0         sub-01  NaN  NaN   NaN     NaN     NaN
1         sub-02  NaN  NaN   NaN     NaN     NaN
2         sub-03  NaN  NaN   NaN     NaN     NaN
3         sub-04  NaN  NaN   NaN     NaN     NaN
4         sub-06  NaN  NaN   NaN     NaN     NaN
PAT path exists: True
Found FreeSurfer subject PAT with directories:
['mri', 'tmp', 'stats', 'surf_old', 'touch', 'trash', 'surf', 'voxeloc', 'elec_recon_pierre_voxeloc', 'label', 'bem', 'mri_old', 'scripts', 'elec_recon', 'label_old', 'elec_recon_old']

Processing 02 → FreeSurfer subject: PAT_3455
FreeSurfer directory: /media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions/PAT_3455

--- Traitement TRC ---
Date:  2022-06-18 09:29:27
Creating RawArray with float64 data, n_channels=121, n_times=8601728
    Range : 0 ... 8601727 =      0.000 ...  4200.062 secs
Ready.
sEEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=1, n_times=86017

/home/aboschun/MIPlab-Project/loading_PAT3455.py:254: RuntimeWarning: Fiducial point nasion not found, assuming identity unknown to head transformation
  raw.set_montage(montage)
/home/aboschun/MIPlab-Project/loading_PAT3455.py:254: RuntimeWarning: Not setting positions of 4 ecg/stim channels found in montage:
['MKR1+', 'photodiode', 'MKR2+', 'ECG']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw.set_montage(montage)
/home/aboschun/MIPlab-Project/loading_PAT3455.py:265: RuntimeWarning: This filename (/home/aboschun/MIPlab-Project/sub-02_montage.fif) does not conform to MNE naming conventions. All montage files should end with -dig.fif or -dig.fif.gz
  montage.save(montage_path, overwrite=True)


Could not get volume labels: 'AD1'

Creating brain visualization for PAT_3455


2026-05-17 15:46:29.224 (  32.150s) [    7F839ADE84C0]vtkXOpenGLRenderWindow.:1460  WARN| bad X server connection. DISPLAY=


Channel types::	seeg: 116
Added electrodes to brain
Montage exists: True
Saved: /home/aboschun/MIPlab-Project/figures/02/electrodes/02_electrodes_lateral_right.png
Saved: /home/aboschun/MIPlab-Project/figures/02/electrodes/02_electrodes_lateral_left.png
Saved: /home/aboschun/MIPlab-Project/figures/02/electrodes/02_electrodes_top.png
Saved: /home/aboschun/MIPlab-Project/figures/02/electrodes/02_electrodes_front.png
Saved: /home/aboschun/MIPlab-Project/figures/02/electrodes/02_electrodes_back.png

All images saved to /home/aboschun/MIPlab-Project/figures/02
Overwriting existing file.
Writing /home/aboschun/MIPlab-Project/data/raw/raw_sub-02.fif
Overwriting existing file.


/home/aboschun/MIPlab-Project/loading_PAT3455.py:390: RuntimeWarning: This filename (/home/aboschun/MIPlab-Project/data/raw/raw_sub-02.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(raw_save_path, overwrite=True)


Closing /home/aboschun/MIPlab-Project/data/raw/raw_sub-02.fif
[done]
Saved raw to /home/aboschun/MIPlab-Project/data/raw/raw_sub-02.fif


In [5]:
# If wanted, compare the BIDS and TRC raw files to verify they are consistent
# Annotations fail if not executed
if data_source == "TRC" or data_source == "Check":
    raw.set_annotations(annotations)
    # if data_source == "Check":
        # Compare BIDS and TRC raw files
        # print(raw_bids)
        # print(raw_trc)
        # # Verify what actually matters
        # print(f"Sfreq:    BIDS={raw_bids.info['sfreq']}, TRC={raw_trc.info['sfreq']}")
        # print(f"N times:  BIDS={len(raw_bids.times)}, TRC={len(raw_trc.times)}")
        # print(f"N ch:     BIDS={len(raw_bids.ch_names)}, TRC={len(raw_trc.ch_names)}")
        # print(f"Ch match: {raw_bids.ch_names == raw_trc.ch_names}")
        # print(f"Ch types match: {raw_bids.get_channel_types() == raw_trc.get_channel_types()}")

#### Check montage's electrodes positions and visualize them (all electrodes):

In [6]:
# Get transform and figure
trans = mne.transforms.Transform('mri', 'head')
fig = mne.viz.plot_alignment(
    raw.info,
    trans=trans,         
    subject=fs_subject,
    subjects_dir=subjects_dir,
    surfaces=[],
    coord_frame="mri",
)

# Get picks and brain volume
picks_hip = mne.pick_channels(raw.info['ch_names'], include=montage.ch_names)

brain = mne.viz.Brain(
    fs_subject,
    alpha=0.1,
    cortex="low_contrast",
    subjects_dir=str(fs_subjects_dir), 
    units="m",
    figure=fig,
)
#brain.add_volume_labels(aseg="aparc+aseg", labels=regions_of_interest)

# Add electrode labels
info_hip = mne.pick_info(raw.info, sel=picks_hip)
brain.add_sensors(info_hip, trans=trans)
for idx in picks_hip:
    ch = raw.info['chs'][idx]
    pos = ch['loc'][:3]  # x, y, z in head coords
    brain.plotter.add_point_labels(
        points=[pos],
        labels=[raw.info['ch_names'][idx]],
        point_size=0,
        font_size=12,
        text_color="white",
        shape=None,
        render=False,
    )

# Set and Save views
distance = 0.1
focalpoint = (0.03,-0.0311,0.001)

brain.show_view(azimuth=120, elevation=90)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_alldata01.png")

brain.show_view(azimuth=60, elevation=90)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_alldata02.png")

brain.show_view(azimuth=60, elevation=45)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_alldata03.png")

brain.show_view(azimuth=0, elevation=45)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_alldata04.png")

brain.show_view(azimuth=30, elevation=45)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_alldata05.png")

Channel types::	seeg: 116


A view with name (P_0x7f80fc8add10_0) is already registered
 => returning previous one


Channel types::	seeg: 116


#### Keep electrodes that are relevant to our study (hippocampus region) :

Important cell where we choose the electrode we will use for our analysis

In [7]:
aseg = "aparc+aseg"  # parcellation/anatomical segmentation atlas
path_atlas = "/media/RCPNAS/sEEG_MARS_Alison/sourcedata/reconstructions"

# Get volume labels for each electrode based on the montage and FreeSurfer subject
labels, colors = mne.get_montage_volume_labels(
    montage, patient, subjects_dir=path_atlas, aseg=aseg
)
print(f"Labels: {labels}")

# Separate by electrodes which have names like LAMY 1
electrodes = set(
    [
        "".join([lttr for lttr in ch_name if not lttr.isdigit() and lttr != " "])
        for ch_name in montage.ch_names
    ]
)
print(f"Electrodes in the dataset: {electrodes}")

# Define regions of interest (ROIs) to keep
regions_of_interest = {
    "ctx-rh-parahippocampal",
    "ctx-rh-fusiform",
    "Right-Hippocampus",
    "ctx-lh-parahippocampal",
    "ctx-lh-fusiform",
    "Left-Hippocampus",
}

# Get labels that correspond to the ROIs --> electrodes that are in the regions of interest
filtered_labels = {
    ch: region_list
    for ch, region_list in labels.items()
    if any(r in regions_of_interest for r in region_list)
}

print(f"Kept {len(filtered_labels)}/{len(labels)} electrodes")
for ch, regions in filtered_labels.items():
    if "Right-Cerebral-White-Matter" in regions:
        regions.remove("Right-Cerebral-White-Matter")
    if "Left-Cerebral-White-Matter" in regions:
        regions.remove("Left-Cerebral-White-Matter")
    print(f"  {ch}: {regions}")


# New set of electrodes that have labels in the ROIs
filtered_electrodes = set([ch.split(" ")[0] for ch in filtered_labels.keys()])
filtered_channels = list(filtered_labels.keys()) 

# Plot their ROIs
fig, ax = mne.viz.plot_channel_labels_circle(filtered_labels, colors, picks=list(filtered_electrodes))
fig.text(0.3, 0.9, "Anatomical Labels", color="white")
fig.savefig(f"figures/{subject}/electrodes/{subject}_allelec_labels.png", dpi=300)

Labels: OrderedDict({'AD1': ['Right-Cerebral-White-Matter', 'Right-Amygdala'], 'AD2': ['Right-Cerebral-White-Matter'], 'AD3': ['Right-Cerebral-White-Matter'], 'AD4': ['Right-Cerebral-White-Matter', 'ctx-rh-middletemporal'], 'AD5': ['Right-Cerebral-White-Matter', 'ctx-rh-middletemporal'], 'AD6': ['Right-Cerebral-White-Matter', 'ctx-rh-middletemporal'], 'AD7': [], 'AD8': [], 'HAD1': ['Right-Cerebral-White-Matter', 'Right-Hippocampus'], 'HAD2': ['Right-Cerebral-White-Matter', 'Right-Hippocampus'], 'HAD3': ['Right-Cerebral-White-Matter', 'Right-Inf-Lat-Vent', 'Right-Hippocampus'], 'HAD4': ['Right-Cerebral-White-Matter', 'Right-Inf-Lat-Vent', 'Right-Hippocampus', 'ctx-rh-fusiform'], 'HAD5': ['Unknown', 'Right-Cerebral-White-Matter', 'Right-Inf-Lat-Vent'], 'HAD6': ['Right-Cerebral-White-Matter'], 'HAD7': ['Right-Cerebral-White-Matter'], 'HAD8': ['Right-Cerebral-White-Matter', 'ctx-rh-inferiortemporal'], 'HAD9': ['Right-Cerebral-White-Matter', 'ctx-rh-middletemporal'], 'HAD10': ['Right-Cerebr

## <span style="color:aquamarine">Filter the raw data</span>

This cell will filter the raw data through a standard procedure:
- Bandpass filter between 0.1Hz and 250Hz
- Remove power line noise and harmonics

In [8]:
raw_filtered = raw.copy().pick_channels(filtered_electrodes) # TODO : test with all electrodes
# Filter raw signals
raw_filtered.load_data()

# band-pass filter 0.1 < 250 Hz
raw_filtered.filter(.1, 250, picks='seeg', fir_design='firwin', n_jobs=9)

# remove power line 50 Hz and harmonics
for ifreq in np.arange(50, 251, 100): # TODO: might be sufficient to filter every 100Hz
    raw_filtered.notch_filter(ifreq, 
                     picks='seeg', 
                     notch_widths=4,
                     method='iir', 
                     iir_params={'order':6, 'ftype':'butter'},
                     n_jobs=9)

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 2.5e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 250.00 Hz
- Upper transition bandwidth: 62.50 Hz (-6 dB cutoff frequency: 281.25 Hz)
- Filter length: 67585 samples (33.000 s)



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    6.1s
[Parallel(n_jobs=9)]: Done  13 out of  17 | elapsed:    6.4s remaining:    2.0s
[Parallel(n_jobs=9)]: Done  17 out of  17 | elapsed:    6.5s finished


Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 48 - 52 Hz

IIR filter parameters
---------------------
Butterworth bandstop zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 24 (effective, after forward-backward)
- Cutoffs at 47.50, 52.50 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    1.3s
[Parallel(n_jobs=9)]: Done  13 out of  17 | elapsed:    1.5s remaining:    0.5s
[Parallel(n_jobs=9)]: Done  17 out of  17 | elapsed:    1.7s finished


Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 1.5e+02 - 1.5e+02 Hz

IIR filter parameters
---------------------
Butterworth bandstop zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 24 (effective, after forward-backward)
- Cutoffs at 147.50, 152.50 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    1.2s
[Parallel(n_jobs=9)]: Done  13 out of  17 | elapsed:    1.4s remaining:    0.4s


Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 2.5e+02 - 2.5e+02 Hz

IIR filter parameters
---------------------
Butterworth bandstop zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 24 (effective, after forward-backward)
- Cutoffs at 247.50, 252.50 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Done  17 out of  17 | elapsed:    1.6s finished
[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.9s
[Parallel(n_jobs=9)]: Done  13 out of  17 | elapsed:    1.1s remaining:    0.3s
[Parallel(n_jobs=9)]: Done  17 out of  17 | elapsed:    1.2s finished


In [9]:
# Sanity check
raw_filtered.annotations

<Annotations | 102 segments: Always (20), Current (17), Distant Futur ...>

## <span style="color:aquamarine">Map the events to the corresponding conditions</span>

For each stimulus exposition, we need to link it to the corresponding condition (current, past, ...):

In [10]:
if data_source == "BIDS":
    print("BIDS")
    annot_df = raw_filtered.annotations.to_data_frame()
    # Create event id from macthing description and condition
    print(annot_df['description'])
    print(annot_df['condition'])

    event_id_for_events = {}
    for desc, cond in zip(annot_df['description'], annot_df['condition']):
        if cond not in event_id_for_events and cond in ['Current', 'Past', 'Distant Past', 'Futur', 'Distant Futur', 'Always', 'Never']:
            event_id_for_events[str(desc)] = int(desc)  # Assign a unique integer ID
    print(f"Event ID mapping: {event_id_for_events}")

    events, _ = mne.events_from_annotations(
        raw_filtered, 
        event_id={k: v for k, v in event_id_for_events.items()}
    )

    # Change events to correct conditions
    event_id = {}
    for desc, cond in zip(annot_df['description'], annot_df['condition']):
        if cond not in event_id and cond in ['Current', 'Past', 'Distant Past', 'Futur', 'Distant Futur', 'Always', 'Never']:
            event_id[str(cond)] = int(desc)  # Assign a unique integer ID
    
    #print(f"Event ID mapping: {event_id}")
    #print(f"Events : {events}")

elif data_source == "TRC" or data_source == "Check":
    print("TRC")
    annot_df = df.copy()
    valid_conds = annot_df['condition'].unique() # Get condition labels in appearing order
    event_id = {str(cond): i+1 for i, cond in enumerate(valid_conds)} # Assign ids based on condition labels

    events, _ = mne.events_from_annotations(
        raw_filtered, 
        event_id={k: v for k, v in event_id.items()}
    )
    
    

TRC
Used Annotations descriptions: [np.str_('Always'), np.str_('Current'), np.str_('Distant Futur'), np.str_('Distant Past'), np.str_('Futur'), np.str_('Never'), np.str_('Past')]


In [11]:
# Check length of events
if data_source == "TRC" or data_source == "Check":
    print("TRC")
    t0 = annot_df['stim_onset'].iloc[0]
    tf = annot_df['stim_onset'].iloc[-1]
    print((tf-t0).total_seconds()) # for patient 3455: 1497.12341 seconds
if data_source == "BIDS":
    print("BIDS")
    t0 = annot_df['onset'].iloc[0]
    tf = annot_df['onset'].iloc[-1]
    print((tf-t0).total_seconds()) # for patient 3455: 1538.289062 seconds

TRC
1497.12341


In [12]:
#epochs = mne.Epochs(raw_filtered, events=events, event_id=event_id, detrend=1, baseline=None, tmin=-5, tmax=10, preload=True)
# Check data to understand the square signal
#data, times = epochs["Current"].get_data(picks="TOD2").squeeze(), epochs.times #raw_filtered["TOD2"]
#tmin, tmax = 0, 3200
#plt.figure()
#plt.plot(times, data.squeeze().mean(axis=0)) #times[tmin:tmax], data.squeeze()[tmin:tmax]
#plt.savefig("figures/{subject}/raw_signal_example.png", dpi=300)

## <span style="color:aquamarine">Epoching and Re-referencing</span>

Let's separate the average evoked response for the 7 different conditions :

In [13]:
from epoching import get_epochs_from_raw_filtered

In [14]:
epochs_ref, epochs_ref_no_hga, band_mean_amps, envelopes = get_epochs_from_raw_filtered(raw_filtered, events, event_id, tmin, tmax, subject=subject, reference=reference, pad=PAD)

cond_current = epochs_ref['Current'].average()
cond_never = epochs_ref['Never'].average()
cond_always = epochs_ref['Always'].average()
cond_past = epochs_ref['Past'].average()
cond_futur = epochs_ref['Futur'].average()
cond_distant_past = epochs_ref['Distant Past'].average()
cond_distant_futur = epochs_ref['Distant Futur'].average()

Not setting metadata
102 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 102 events and 18433 original time points ...
0 bad epochs dropped
Bipolar Referencing
sEEG channel type selected for re-referencing
Not setting metadata
102 matching events found
No baseline correction applied
0 projection items activated
Added the following bipolar channels:
TOD2-TOD3, TOD3-TOD4, PHD1-PHD2, PHD2-PHD3, PHD3-PHD4, PHD4-PHD5, HPD1-HPD2, HPD2-HPD3, HPD3-HPD4, HPD4-HPD5, HAD1-HAD2, HAD2-HAD3, HAD3-HAD4
Setting up band-pass filter from 70 - 80 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 70.00, 80.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 833 tasks      | elapsed:    0.4s
[Parallel(n_jobs=9)]: Done 1318 out of 1326 | elapsed:    0.7s remaining:    0.0s
[Parallel(n_jobs=9)]: Done 1326 out of 1326 | elapsed:    0.7s finished


Setting up band-pass filter from 80 - 90 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 80.00, 90.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 1081 tasks      | elapsed:    0.6s
[Parallel(n_jobs=9)]: Done 1326 out of 1326 | elapsed:    0.8s finished


Setting up band-pass filter from 90 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 90.00, 100.00 Hz: -6.02, -6.02 dB



...

Setting up band-pass filter from 1.9e+02 - 2e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 190.00, 200.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 1019 tasks      | elapsed:    0.7s
[Parallel(n_jobs=9)]: Done 1326 out of 1326 | elapsed:    0.9s finished


Setting up band-pass filter from 2e+02 - 2.1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 200.00, 210.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 1019 tasks      | elapsed:    0.7s
[Parallel(n_jobs=9)]: Done 1326 out of 1326 | elapsed:    0.9s finished


Computed HGA envelope: (102, 13, 18433)  →  (epochs, channels, times)


In [15]:
# This cell is to verify that the normalization worked as intended, 
# by comparing the mean amplitude per band before and after normalization, 
# and by plotting the raw PSD to see the effect of the notch filters and the HGA band selection.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
elec = "TOD2-TOD3"
band_centers = np.arange(70, 201, 10)
ch_idx = epochs_ref.ch_names.index(elec)

# Before normalization — raw mean amplitudes per band
band_means_raw = band_mean_amps[:, :, ch_idx, 0].mean(axis=1)

raw_data = epochs_ref_no_hga.copy().crop(tmin=tmin, tmax=tmax).get_data(picks=[elec])[:, 0, :]  # (n_epochs, n_times)
freqs_psd = np.fft.rfftfreq(raw_data.shape[-1], d=1/epochs_ref.info['sfreq'])
psd_raw = np.abs(np.fft.rfft(raw_data, axis=-1)).mean(axis=0) ** 2

# After normalization — mean amplitude per band from normalized envelopes
# Recompute from the stored envelopes array before averaging
band_means_norm = []
for b in range(len(envelopes)):  # envelopes shape: (n_bands, n_epochs, n_ch, n_times)
    band_means_norm.append(envelopes[b, :, ch_idx, :].mean())
band_means_norm = np.array(band_means_norm)

axes[0].plot(band_centers, band_means_raw / band_means_raw.max(), 'o-', color='blue', label='Before normalization')
axes[0].plot(band_centers, band_means_norm / band_means_norm.max(), 'o-', color='red', label='After normalization')
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Normalized mean amplitude")
axes[0].set_title("Per-band amplitude before vs after\n1/f normalization")
axes[0].legend()
axes[0].axvspan(140, 160, alpha=0.1, color='orange', label='150Hz notch region')

# Raw PSD with notch regions highlighted
axes[1].semilogy(freqs_psd, psd_raw)
axes[1].set_xlim(1, 250)
axes[1].axvspan(45, 55, alpha=0.2, color='red', label='50Hz notch')
axes[1].axvspan(145, 155, alpha=0.2, color='orange', label='150Hz notch')
axes[1].axvspan(70, 200, alpha=0.1, color='blue', label='HGA band')
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Power (log scale)")
axes[1].set_title("Raw PSD — notch regions")
axes[1].legend()

fig.suptitle(f"Normalization verification — {elec}", fontsize=13)
plt.tight_layout()
fig.savefig(f"figures/{subject}/test_new_filtering/{subject}_{reference}_normalization_verification.png", dpi=300)
plt.close(fig)

In [16]:
# Plot response of a few electrodes to check the effect of the normalization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
electrodes_to_plot = ['PHD2-PHD3', 'PHD3-PHD4', 'TOD2-TOD3', 'TOD3-TOD4', 'HPD2-HPD3', 'HAD2-HAD3']
for i, elec in enumerate(electrodes_to_plot):
    if elec in epochs_ref.ch_names:
        evoked = epochs_ref.copy().crop(tmin=-5, tmax=10).average(picks=[elec])
        axes[i//3, i%3].plot(evoked.times, evoked.get_data(picks=[elec])[0])
        axes[i//3, i%3].axvline(0, color='r', linestyle='--', linewidth=0.8)
        axes[i//3, i%3].set_title(f"{elec} — HFA envelope")
        axes[i//3, i%3].set_xlabel("Time (s)")
        axes[i//3, i%3].set_ylabel("Amplitude (µV)")
    else:
        axes[i//3, i%3].text(0.5, 0.5, f"{elec} not in data", horizontalalignment='center', verticalalignment='center')
        axes[i//3, i%3].set_axis_off()
fig.suptitle(f"HFA envelope response — {reference} referencing", fontsize=16)
plt.tight_layout()
fig.savefig(f"figures/{subject}/test_new_filtering/{subject}_{reference}_envelope_response.png", dpi=300)
plt.close(fig)

/tmp/ipykernel_1226748/2313659542.py:6: RuntimeWarning: tmin is not in time interval. tmin is set to <class 'mne.epochs.Epochs'>.tmin (-1 s)
  evoked = epochs_ref.copy().crop(tmin=-5, tmax=10).average(picks=[elec])
/tmp/ipykernel_1226748/2313659542.py:6: RuntimeWarning: tmax is not in time interval. tmax is set to <class 'mne.epochs.Epochs'>.tmax (7 s)
  evoked = epochs_ref.copy().crop(tmin=-5, tmax=10).average(picks=[elec])
/tmp/ipykernel_1226748/2313659542.py:6: RuntimeWarning: tmin is not in time interval. tmin is set to <class 'mne.epochs.Epochs'>.tmin (-1 s)
  evoked = epochs_ref.copy().crop(tmin=-5, tmax=10).average(picks=[elec])
/tmp/ipykernel_1226748/2313659542.py:6: RuntimeWarning: tmax is not in time interval. tmax is set to <class 'mne.epochs.Epochs'>.tmax (7 s)
  evoked = epochs_ref.copy().crop(tmin=-5, tmax=10).average(picks=[elec])
/tmp/ipykernel_1226748/2313659542.py:6: RuntimeWarning: tmin is not in time interval. tmin is set to <class 'mne.epochs.Epochs'>.tmin (-1 s)
 

WARNING: 
After applying the Hilbert transform on HGA frequencies, the meaning frequency domain of the signal is not the same as before. The enveloppe of the signal will give me the modulation of the signal from HGA frequencies

Plotting some elctrodes to see the response

In [17]:
# TODO: Plot responses of all concerned electrode for one condition in one plot to see which ones are relevant
if reference == "bipolar": 
    if subject == "01":
        electrodes_to_plot = ["HAG1-HAG2", "HAG2-HAG3", "HAG3-HAG4", "HAG4-HAG5", "HPG1-HPG2", "HPG2-HPG3", "HPG3-HPG4", "PHG1-PHG2", "PHG2-PHG3", "PHG3-PHG4", "PHG4-PHG6", "TOL2-TOL5"]

    if subject == "02":
        electrodes_to_plot = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"] 

if reference == "bipolar_shaft":
    electrodes_to_plot = ["TOD2-PHD1", "TOD2-PHD2", "TOD3-PHD3", "TOD4-PHD4", "TOD4-PHD5", "PHD1-HPD1", "PHD2-HPD2", "PHD3-HPD3", "PHD4-HPD4", "PHD5-HPD5", "HPD1-HAD1", "HPD2-HAD2", "HPD3-HAD2", "HPD3-HAD3", "HPD4-HAD3", "HPD5-HAD4" ]

if reference == "average":
    electrodes_to_plot = ["TOD2", "TOD3", "TOD4", "PHD1", "PHD2", "PHD3", "PHD4", "PHD5", "HPD1", "HPD2", "HPD3", "HPD4", "HPD5", "HAD1", "HAD2", "HAD3", "HAD4"]              

# Plot for condition Current
evoked = cond_current
data = evoked.get_data(picks=electrodes_to_plot)
times = cond_current.times

n_elec = len(electrodes_to_plot)
fig, axes = plt.subplots(n_elec, 1, figsize=(12, 2.5 * n_elec), sharex=True)

for ax, elec, trace in zip(axes, electrodes_to_plot, data):
    ax.plot(times, trace)
    ax.axvline(0, color='r', linestyle='--', linewidth=0.8)
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    ax.set_title(elec, fontsize=9, loc='left')
    ax.set_ylabel("dB re. baseline", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"All electrodes — Current", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(f"figures/{subject}/electrodes/{subject}_all_electrodes_Current_{reference}_evoked.png", dpi=300)
plt.close(fig)

## <span style="color:aquamarine">Bad trials removal</span>

Remove bad trials, including technical outliers (amplitude too high) and behaviorally wrong answers (already removed when loading the data).

In [18]:
# Plot all trials for one electrode to check variability
for elec in electrodes_to_plot:
    if elec in epochs_ref.ch_names:
        data = epochs_ref['Current'].get_data(picks=[elec])[:, 0, :] * 1e6  # convert V → µV
        times = epochs_ref.times
        evoked_data = data.mean(axis=0)

        fig, ax = plt.subplots(figsize=(12, 6))
        for trial in data:
            ax.plot(times, trial, color='gray', alpha=0.3, linewidth=0.5)
        ax.plot(times, evoked_data, color='red', linewidth=2, label='Average')
        ax.axvline(0, color='r', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='k', linestyle=':', linewidth=0.5)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude (µV)")
        ax.set_title(f"{elec} — All trials and average — Current")
        ax.legend()
        fig.savefig(f"figures/{subject}/bad_rejection/{subject}_{elec}_Current_{reference}_all_trials.png", dpi=300)
        plt.close(fig)

In [19]:
def late_artifact_rejection(epochs, threshold_uv=300, late_tmin=1.5, late_tmax=7, picks='seeg'):
    """
    Reject epochs with outlier peaks in the late post-stimulus window,
    where genuine neural responses are unlikely.
    
    Parameters
    ----------
    threshold_uv : float
        Peak-to-peak threshold in µV for the late window
    late_tmin : float
        Start of the late window in seconds (after stimulus onset)
    """
    threshold_v = threshold_uv * 1e-6

    seeg_picks = mne.pick_types(epochs.info, seeg=True)
    seeg_ch_names = [epochs.ch_names[i] for i in seeg_picks]

    # Split into early (response) and late (artifact check) windows
    early_mask = (epochs.times >= 0) & (epochs.times < late_tmin)  # also limit to first 5s to avoid long-lasting responses
    late_mask  = (epochs.times >= late_tmin) & (epochs.times < late_tmax)

    data = epochs.get_data(picks='seeg')  # (n_epochs, n_ch, n_times)

    # P2P in late window only
    late_data = data[:, :, late_mask]
    #late_ptp  = max(late_data.max(axis=-1), np.abs(late_data.min(axis=-1)))  # (n_epochs, n_ch)
    late_max = np.maximum(late_data.max(axis=-1), np.abs(late_data.min(axis=-1)))  # (n_epochs, n_ch)

    # Also compute early P2P for comparison
    early_data = data[:, :, early_mask]
    early_ptp  = early_data.max(axis=-1) - early_data.min(axis=-1)

    # Flag epochs where late P2P exceeds threshold in any channel
    bad_epoch_mask = (late_max > threshold_v).any(axis=-1)
    good_epoch_mask = ~bad_epoch_mask

    print(f"Late window: t >= {late_tmin}s")
    print(f"Threshold: {threshold_uv} µV")
    print(f"Rejected {bad_epoch_mask.sum()} / {len(epochs)} epochs ({bad_epoch_mask.mean()*100:.1f}%)")

    for i in np.where(bad_epoch_mask)[0]:
        bad_chs = [seeg_ch_names[j] for j in range(len(seeg_ch_names))
                   if late_max[i, j] > threshold_v]
        print(f"  Epoch {i}: late artifact in {bad_chs} "
              f"(late P2P={late_max[i].max()*1e6:.0f}µV, "
              f"early P2P={early_ptp[i].max()*1e6:.0f}µV)")

    epochs_clean = epochs[good_epoch_mask]
    original_annotations = epochs.annotations
    good_indices = np.where(good_epoch_mask)[0]

    onsets       = original_annotations.onset[good_indices]
    durations    = original_annotations.duration[good_indices]
    descriptions = original_annotations.description[good_indices]

    clean_annotations = mne.Annotations(
        onset=onsets,
        duration=durations,
        description=descriptions
    )

    epochs_clean.set_annotations(clean_annotations)

    return epochs_clean, good_epoch_mask

threshold_ref = 8.5*1e7
late_tmin = 2
late_tmax = 7

epochs_ref_removed, good_mask = late_artifact_rejection(
    epochs_ref,
    threshold_uv=threshold_ref,   # can be stricter than full-epoch threshold
    late_tmin=late_tmin,       # adjust based on your expected response duration
    late_tmax=late_tmax
)

threshold_no_hga = 2.5*1e2
# TODO: The threshold is not adapted to the not filtered epochs
epochs_ref_no_hga_removed, good_mask = late_artifact_rejection(
    epochs_ref_no_hga,
    threshold_uv=threshold_no_hga,   # can be stricter than full-epoch threshold
    late_tmin=late_tmin,       # adjust based on your expected response duration
    late_tmax=late_tmax
)

for elec in electrodes_to_plot:
    if elec in epochs_ref_removed.ch_names:
        data = epochs_ref_removed['Current'].get_data(picks=[elec])[:, 0, :] * 1e6  # convert V → µV
        times = epochs_ref_removed.times
        evoked_data = data.mean(axis=0)

        fig, ax = plt.subplots(figsize=(12, 6))
        for trial in data:
            ax.plot(times, trial, color='gray', alpha=0.3, linewidth=0.5)
        ax.plot(times, evoked_data, color='red', linewidth=2, label='Average')
        ax.axvline(0, color='r', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='k', linestyle=':', linewidth=0.5)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Amplitude (µV)")
        ax.set_title(f"{elec} — All trials and average Clean — Current")
        ax.legend()
        fig.savefig(f"figures/{subject}/bad_rejection/{subject}_{elec}_Current_{reference}_all_trials_clean.png", dpi=300)
        plt.close(fig)

Late window: t >= 2s
Threshold: 85000000.0 µV
Rejected 6 / 102 epochs (5.9%)
  Epoch 3: late artifact in ['PHD2-PHD3'] (late P2P=86812070µV, early P2P=82419604µV)
  Epoch 4: late artifact in ['HAD1-HAD2'] (late P2P=85115023µV, early P2P=112555872µV)
  Epoch 46: late artifact in ['HAD2-HAD3'] (late P2P=88240727µV, early P2P=90515134µV)
  Epoch 69: late artifact in ['PHD2-PHD3'] (late P2P=86887596µV, early P2P=92486372µV)
  Epoch 70: late artifact in ['PHD2-PHD3'] (late P2P=106735654µV, early P2P=113822967µV)
  Epoch 76: late artifact in ['PHD1-PHD2'] (late P2P=95920690µV, early P2P=89946720µV)
Late window: t >= 2s
Threshold: 250.0 µV
Rejected 5 / 102 epochs (4.9%)
  Epoch 46: late artifact in ['HAD2-HAD3'] (late P2P=310µV, early P2P=194µV)
  Epoch 47: late artifact in ['HAD2-HAD3'] (late P2P=254µV, early P2P=205µV)
  Epoch 50: late artifact in ['HPD1-HPD2', 'HAD2-HAD3'] (late P2P=501µV, early P2P=168µV)
  Epoch 84: late artifact in ['HAD2-HAD3'] (late P2P=313µV, early P2P=251µV)
  Epoch

In [20]:
epochs_ref = epochs_ref_removed
epochs_ref_no_hga = epochs_ref_no_hga_removed

## <span style="color:aquamarine">Envelope visualization and analysis</span>

### <span style="color:orange">HGA analysis</span>

For HGA we will need to use epochs_ref

In [21]:
cond_colors = {
    'Current':       'black',
    'Past':          'steelblue',
    'Distant Past':  'cornflowerblue',
    'Futur':         'tomato',
    'Distant Futur': 'salmon',
}
conds = list(cond_colors.keys())

for ch in electrodes_to_plot:
    fig, ax = plt.subplots(figsize=(14, 4))

    for cond, color in cond_colors.items():
        evoked_hga = epochs_ref[cond].average()
        data = evoked_hga.get_data(picks=[ch])[0]
        n    = len(epochs_ref[cond])

        # SEM across trials
        sem = epochs_ref[cond].get_data(picks=[ch])[:, 0, :].std(axis=0) / np.sqrt(n)

        ax.fill_between(times, data - sem, data + sem, alpha=0.15, color=color)
        ax.plot(times, data, lw=1.8, color=color, label=f'{cond} (n={n})')

    ax.axhline(0, color='k', lw=0.8, ls=':', label='0 µV')
    ax.axvline(0, color='k', lw=1.2, ls='--', label='Stimulus onset')

    ax.set_xlabel('Time (s)', fontsize=11)
    ax.set_ylabel('HGA (% change from baseline)', fontsize=11)
    ax.set_title(f'HGA envelope — {ch}', fontsize=13)
    ax.legend(fontsize=9, ncol=3, loc='upper right')
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(
        f"figures/{subject}/hga_envelopes/{subject}_HGA_envelope_{ch}_all_conds.png",
        dpi=300, bbox_inches='tight'
    )
    plt.close()
    print(f"Saved: {ch}")

Saved: TOD2-TOD3
Saved: TOD3-TOD4
Saved: PHD1-PHD2
Saved: PHD2-PHD3
Saved: PHD3-PHD4
Saved: PHD4-PHD5
Saved: HPD1-HPD2
Saved: HPD2-HPD3
Saved: HPD3-HPD4
Saved: HPD4-HPD5
Saved: HAD1-HAD2
Saved: HAD2-HAD3
Saved: HAD3-HAD4


It seems that HGA spikes right after the stimulus onset for basically all conditions without a clear observable difference between temporal and current conditions.

### <span style="color:orange">Other frequency bands analysis</span>

The next cell will compute the envelope of each of the 5 main frequency bands: 


In [22]:
# Main frequency bands
bands = {
    "theta":     (4,   8),
    "alpha":     (8,  12),
    "beta":      (12, 30),
    "low_gamma": (30, 80),
    #"high_gamma":(80, 200), # already computed in the HGA envelope, and in details
}

epochs_bands = {}

# Conditions to consider
cond_list = ["Current", "Past", "Futur", "Distant Past", "Distant Futur"] #["Current", "Never", "Always", "Past", "Futur", "Distant Past", "Distant Futur"]
for cond in cond_list:
    for band_name, (fmin, fmax) in bands.items():
        # copy the referenced epochs (before any filtering)
        ep = epochs_ref_no_hga[cond].copy()
        ep._data = np.ascontiguousarray(ep._data)
        
        # bandpass to the band of interest
        ep.filter(fmin, fmax, picks='seeg', method="iir", iir_params=None, n_jobs=9)
        
        # extract envelope
        ep.apply_hilbert(envelope=True)
        ep.filter(None, 20, picks='seeg', method="iir", iir_params=None, n_jobs=9)
        
        # baseline normalization
        data = ep.get_data(picks='seeg')
        baseline_mask = ep.times < 0
        baseline_mean = data[:, :, baseline_mask].mean(axis=-1, keepdims=True)
        ep._data[:, :len(ep.ch_names), :] = data - baseline_mean

        epochs_bands[(band_name, cond)] = ep.crop(tmin=tmin, tmax=tmax)

    # Plot evoked envelope per band per electrode
    # for elec in electrodes_to_plot:
    #     fig, axes = plt.subplots(len(bands), 1, figsize=(12, 2.5 * len(bands)), sharex=True)

    #     for ax, (band_name, ep) in zip(axes, epochs_bands.items()):
    #         evoked = ep.average()
    #         trace = evoked.get_data(picks=[elec])[0]
    #         ax.plot(ep.times, trace)
    #         ax.axvline(0, color='r', linestyle='--', linewidth=0.8)
    #         ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    #         ax.set_title(band_name, fontsize=9, loc='left')
    #         ax.set_ylabel("Activation [V]", fontsize=8)

    #     axes[-1].set_xlabel("Time (s)")
    #     fig.suptitle(f"Band envelopes — {elec}", fontsize=14)
    #     plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    #     plt.savefig(f"figures/{subject}/band_test/{reference}/electrodes/{subject}_{elec}_{reference}_{cond}_band_envelopes.png", dpi=300)
    #     plt.close(fig)

Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 162 tasks      | elapsed:    0.2s


Setting up low-pass filter at 20 Hz

IIR filter parameters


[Parallel(n_jobs=9)]: Done 221 out of 221 | elapsed:    0.2s finished


---------------------
Butterworth lowpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 20.00 Hz: -6.02 dB



...

[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 143 out of 143 | elapsed:    0.1s finished


Setting up band-pass filter from 30 - 80 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 30.00, 80.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 143 out of 143 | elapsed:    0.1s finished


Setting up low-pass filter at 20 Hz

IIR filter parameters
---------------------
Butterworth lowpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 20.00 Hz: -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 143 out of 143 | elapsed:    0.1s finished


The next cell will plot the envelope for each condition+electrode per band:

In [23]:
# Plot
colors = {
    "Current":      "#1f77b4",
    "Never":        "#ff7f0e",
    "Always":       "#2ca02c",
    "Past":         "#d62728",
    "Futur":        "#9467bd",
    "Distant Past": "#8c564b",
    "Distant Futur":"#e377c2",
}

for band_name in bands.keys():
    n_elec = len(electrodes_to_plot)
    fig, axes = plt.subplots(n_elec, 1, figsize=(14, 3 * n_elec), sharex=True)

    for ax, elec in zip(axes, electrodes_to_plot):
        for cond in cond_list:
            ep = epochs_bands[(band_name, cond)]
            evoked = ep.average()
            trace = evoked.get_data(picks=[elec])[0]
            ax.plot(ep.times, trace, label=cond, color=colors[cond], linewidth=1.2)

        ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='k', linestyle=':', linewidth=0.5)
        ax.set_title(elec, fontsize=9, loc='left')
        ax.set_ylabel("Activation [V]", fontsize=8)

    axes[0].legend(loc='upper right', fontsize=7, ncol=len(cond_list))
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Band envelope — {band_name} — all conditions", fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f"figures/{subject}/band_test/{reference}/per_band/{subject}_{band_name}_{reference}_all_conditions.png", dpi=300)
    plt.close(fig)

In [24]:
n_bands = len(bands)
n_elec = len(electrodes_to_plot)
fig, axes = plt.subplots(n_bands, n_elec, figsize=(5 * n_elec, 3 * n_bands), sharex=True, sharey='row')

for i, band_name in enumerate(bands.keys()):
    for j, elec in enumerate(electrodes_to_plot):
        ax = axes[i, j]
        for cond in cond_list:
            ep = epochs_bands[(band_name, cond)]
            evoked = ep.average()
            trace = evoked.get_data(picks=[elec])[0]
            ax.plot(ep.times, trace, label=cond, color=colors[cond], linewidth=1.2)

        ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='k', linestyle=':', linewidth=0.5)

        if i == 0:
            ax.set_title(elec, fontsize=10)
        if j == 0:
            ax.set_ylabel(band_name, fontsize=9)
        if i == n_bands - 1:
            ax.set_xlabel("Time (s)", fontsize=8)

axes[0, -1].legend(loc='upper right', fontsize=7, ncol=1)
fig.suptitle(f"Band envelopes — all electrodes — all conditions", fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(f"figures/{subject}/band_test/{reference}/{subject}_{reference}_all_bands_all_electrodes_all_conditions.png", dpi=300)
plt.close(fig)

In [25]:
n_cond = len(cond_list)
fig, axes = plt.subplots(n_elec, n_cond, figsize=(3 * n_cond, 2.5 * n_elec), sharex=True, sharey=True)

for i, elec in enumerate(electrodes_to_plot):
    # Compute global color scale before the loop
    all_traces = []
    for band_name in bands.keys():
        for cond in cond_list:
            ep = epochs_bands[(band_name, cond)]
            evoked = ep.average()
            for el in electrodes_to_plot:
                all_traces.append(evoked.get_data(picks=[el])[0])

    global_max = np.percentile(np.abs(np.array(all_traces)), 95)
    vmin, vmax = -global_max, global_max

    for j, cond in enumerate(cond_list):
        ax = axes[i, j]
        
        # stack all bands for this electrode/condition
        band_traces = []
        for band_name in bands.keys():
            ep = epochs_bands[(band_name, cond)]
            evoked = ep.average()
            trace = evoked.get_data(picks=[elec])[0]
            band_traces.append(trace)
        
        data_matrix = np.array(band_traces)  # (n_bands, n_times)
        
        im = ax.imshow(
            data_matrix,
            aspect='auto',
            origin='lower',
            extent=[ep.times[0], ep.times[-1], 0, len(bands)],
            cmap='RdBu_r',
            vmin=vmin, #-np.percentile(np.abs(data_matrix), 95),
            vmax=vmax #np.percentile(np.abs(data_matrix), 95)
        )
        ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
        ax.set_yticks(np.arange(len(bands)) + 0.5)

        if i == 0:
            ax.set_title(cond, fontsize=9)
        if j == 0:
            ax.set_yticklabels(list(bands.keys()), fontsize=7)
            ax.set_ylabel(elec, fontsize=9)
        else:
            ax.set_yticklabels([])
        if i == n_elec - 1:
            ax.set_xlabel("Time (s)", fontsize=8)

plt.colorbar(im, ax=axes, label="Activation [V]", shrink=0.5)
fig.suptitle(f"Band envelopes — all electrodes — all conditions", fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(f"figures/{subject}/band_test/{reference}/{subject}_{reference}_heatmap.png", dpi=300)
plt.close(fig)

/tmp/ipykernel_1226748/3665752558.py:54: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.03, 1, 0.95])


The next cell will plot for each electrode, its envelope for the different frequency bands:

In [26]:
for elec in electrodes_to_plot:
    n_bands = len(bands)
    fig, axes = plt.subplots(n_bands, 1, figsize=(14, 3 * n_bands), sharex=True)

    for ax, band_name in zip(axes, bands.keys()):
        for cond in cond_list:
            ep = epochs_bands[(band_name, cond)]
            evoked = ep.average()
            trace = evoked.get_data(picks=[elec])[0]
            ax.plot(ep.times, trace, label=cond, color=colors[cond], linewidth=1.2)

        ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='k', linestyle=':', linewidth=0.5)
        ax.set_title(band_name, fontsize=9, loc='left')
        ax.set_ylabel("Activation [V]", fontsize=8)

    axes[0].legend(loc='upper right', fontsize=7, ncol=len(cond_list))
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"Band envelopes — {elec} — all conditions", fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f"figures/{subject}/band_test/{reference}/{subject}_{elec}_{reference}_all_bands_all_conditions.png", dpi=300)
    plt.close(fig)

### <span style="color:orange">Electrode Sanity Check</span>

Here we will visually check that the electrodes do not present undesired behaviours such as drifting.

In [27]:
# TODO: Move this section earlier in the pipeline

In [28]:
suspicious_elec = electrodes_to_plot#["TOD2-TOD3", "PHD1-PHD2", "HAD3-HAD4"]

fig, axes = plt.subplots(len(suspicious_elec), 1, figsize=(20, 4 * len(suspicious_elec)), sharex=True)

for ax, elec in zip(axes, suspicious_elec):
    # 
    epochs_for_plot = epochs_ref_no_hga.copy().crop(tmin=tmin, tmax=tmax)  # crop to focus on the time window of interest

    # Get channel index in epochs 
    ch_idx = epochs_for_plot.ch_names.index(elec)
    
    # Get all trials, all times for this channel
    data = epochs_for_plot.get_data(picks=[elec])  # (n_epochs, 1, n_times)
    
    # Plot each trial in light gray
    for trial in data:
        ax.plot(epochs_for_plot.times, trial[0] * 1e6, color='gray', alpha=0.2, linewidth=0.5)
    
    # Plot the average in color
    ax.plot(epochs_for_plot.times, data.mean(axis=0)[0] * 1e6, color='red', linewidth=1.5, label='average')
    
    ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
    ax.axhline(0, color='k', linestyle=':', linewidth=0.5)
    ax.set_ylabel("Amplitude (µV)", fontsize=9)
    ax.set_title(elec, fontsize=10)
    ax.legend(fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Raw signal — suspicious electrodes", fontsize=14)
plt.tight_layout()
plt.savefig(f"figures/{subject}/band_test/{reference}/{subject}_{reference}_suspicious_electrodes_raw.png", dpi=300)
plt.close(fig)

Same as previous cell but condition dependant:

In [29]:
suspicious_elec = electrodes_to_plot#["TOD2-TOD3", "PHD1-PHD2", "HAD3-HAD4"]

for cond in cond_list:
    fig, axes = plt.subplots(len(suspicious_elec), 1, figsize=(20, 4 * len(suspicious_elec)), sharex=True)
    for ax, elec in zip(axes, suspicious_elec):
        epochs_for_plot = epochs_ref[cond].copy().crop(tmin=tmin, tmax=tmax)

        # Get channel index in epochs_ref
        ch_idx = epochs_for_plot.ch_names.index(elec)
        
        # Get all trials, all times for this channel
        data = epochs_for_plot.get_data(picks=[elec])  # (n_epochs, 1, n_times)
        
        # Plot each trial in light gray
        for trial in data:
            ax.plot(epochs_for_plot.times, trial[0] * 1e6, color='gray', alpha=0.2, linewidth=0.5)
        
        # Plot the average in color
        ax.plot(epochs_for_plot.times, data.mean(axis=0)[0] * 1e6, color='red', linewidth=1.5, label='average')
        
        ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='k', linestyle=':', linewidth=0.5)
        ax.set_ylabel("Amplitude (µV)", fontsize=9)
        ax.set_title(elec, fontsize=10)
        ax.legend(fontsize=8)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle("Raw signal — suspicious electrodes", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"figures/{subject}/band_test/{reference}/suspect/{subject}_{reference}_suspicious_electrodes_raw_{cond}.png", dpi=300)
    plt.close(fig)

Checking that normalizing does not change the response:

In [30]:
suspicious_elec = electrodes_to_plot#["TOD2-TOD3", "PHD1-PHD2", "HAD3-HAD4"]

fig, axes = plt.subplots(len(suspicious_elec), 2, figsize=(16, 4 * len(suspicious_elec)), sharex=True)

for i, elec in enumerate(suspicious_elec):
    epochs_for_plot = epochs_ref.copy().crop(tmin=tmin, tmax=tmax)  # crop to focus on the time window of interest
    # Raw (no normalization)
    data_raw = epochs_for_plot.get_data(picks=[elec])  # (n_epochs, 1, n_times)
    grand_avg_raw = data_raw.mean(axis=0)[0] * 1e6

    # Normalized
    baseline_mask = epochs_for_plot.times < 0
    baseline_mean = data_raw[:, :, baseline_mask].mean(axis=-1, keepdims=True)
    data_norm = (data_raw - baseline_mean) * 1e6
    grand_avg_norm = data_norm.mean(axis=0)[0]

    for j, (grand_avg, data, title) in enumerate([
        (grand_avg_raw, data_raw * 1e6, "Raw (no normalization)"),
        (grand_avg_norm, data_norm,     "Baseline subtracted")
    ]):
        ax = axes[i, j]
        for trial in data:
            ax.plot(epochs_for_plot.times, trial[0], color='gray', alpha=0.15, linewidth=0.4)
        ax.plot(epochs_for_plot.times, grand_avg, color='red', linewidth=1.5, label=f'grand avg (n={len(data)})')
        ax.axvline(0, color='k', linestyle='--', linewidth=0.8)
        ax.axhline(0, color='k', linestyle=':', linewidth=0.5)
        ax.set_ylabel("Amplitude (µV)", fontsize=9)
        if i == 0:
            ax.set_title(title, fontsize=10)
        if j == 0:
            ax.set_ylabel(f"{elec}\nAmplitude (µV)", fontsize=9)
        if i == len(suspicious_elec) - 1:
            ax.set_xlabel("Time (s)")
        ax.legend(fontsize=7)

fig.suptitle("Raw vs baseline-normalized — suspicious electrodes", fontsize=14)
plt.tight_layout()
plt.savefig(f"figures/{subject}/band_test/{reference}/suspect/{subject}_{reference}_normalization_check.png", dpi=300)
plt.close(fig)

### <span style="color:orange">TFR + ITC</span>

This section needs some improvement, it should do the same as with the previous broad frequency bands but with much smaller (higher resolution) frequency bands:

In [31]:
freqs = np.logspace(np.log10(4), np.log10(200), 40)  # 40 freqs from 4-200Hz
n_cycles = freqs / 2  # adaptive — more cycles at high freqs

tfr_avg, itc = epochs_ref.copy().compute_tfr(
    method='morlet', freqs=freqs, n_cycles=n_cycles,
    picks='seeg', return_itc=True, average=True
)
# High ITC + high power = evoked
# High power + low ITC = induced (HGA typically falls here)

Condition dependency:

In [32]:
for cond in cond_list:
    tfr_avg, itc = epochs_ref[cond].copy().compute_tfr(
        method='morlet', freqs=freqs, n_cycles=n_cycles,
        picks='seeg', return_itc=True, average=True
    )
    tfr_avg.crop(tmin=tmin,tmax=tmax).plot(picks=["TOD2-TOD3"], baseline=(-1, 0), mode='logratio', title=f"TFR (average) — TOD2-TOD3")
    plt.savefig(f"figures/{subject}/tfr/{reference}/{subject}_tfr_{cond}.png", dpi=300)
    itc.crop(tmin=tmin,tmax=tmax).plot(picks=["TOD2-TOD3"], baseline=(-1, 0), mode='logratio', title=f"ITC (average) — TOD2-TOD3")
    plt.savefig(f"figures/{subject}/tfr/{reference}/{subject}_itc_{cond}.png", dpi=300)
    plt.close()


Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)


In [33]:
itc.plot(picks=['TOD2-TOD3'], title="ITC — TOD2-TOD3")
plt.savefig(f"figures/{subject}/band_test/{reference}/{subject}_tfr_itc.png", dpi=300)
tfr_avg.plot(picks=['TOD2-TOD3'], title="TFR (power) — TOD2-TOD3")
plt.savefig(f"figures/{subject}/tfr/{reference}/{subject}_tfr_itc_power.png", dpi=300)
plt.close()

No baseline correction applied


No baseline correction applied


Test

In [34]:
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import cross_val_score

# X_all = []
# for ch_index in range(len(epochs_ref.ch_names)):
#     X = mean_envelope[:, ch_idx, :]  # (n_trials, n_times)
#     print(X.shape)
#     X_all.append(X)

# X_all = np.concatenate(X_all, axis=1)  # (n_trials, n
# y = epochs_ref.events[:, 2]  # condition labels
# print(X_all.shape)

# clf = RandomForestClassifier(n_estimators=100, random_state=42)
# scores = cross_val_score(clf, X_all, y, cv=5)
# print(f"Decoding accuracy = {scores.mean()}:.2f ± {scores.std}:.2f")

In [35]:
# event_id

In [36]:
# X_all.shape

In [37]:
# y = epochs_ref.events[:, 2]
# y.shape

In [38]:
# n_trials, n_channels, n_times = mean_envelope.shape

In [39]:
# X_all = mean_envelope.reshape(n_trials, n_channels * n_times)
# X_all = np.delete(X_all, (0), axis=0)
# print(X_all.shape)

In [40]:
# # Reshape to (n_trials, n_channels * n_times)
# X_all = mean_envelope.reshape(n_trials, n_channels * n_times)
# y = epochs_ref.events[:, 2]
# indices_to_remove = np.where((y == 3) | (y == 5))[0]
# X_all = np.delete(X_all, indices_to_remove, axis=0)
# y = np.delete(y, indices_to_remove)

# for trial in range(len(y)):
#     if y[trial] == 2 or y[trial] == 4:
#         y[trial] = 6
#     if y[trial] == 7 :
#         y[trial] = 6

# print(y)
# print(X_all.shape)  # (n_trials, n_channels * n_times)

# clf = RandomForestClassifier(n_estimators=50, random_state=42)
# scores = cross_val_score(clf, X_all, y, cv=5)
# print(f"Decoding accuracy = {scores.mean():.2f} ± {scores.std():.2f}")

## <span style="color:aquamarine">Statistical Analysis</span>

### <span style="color:orange">HGA statistical analysis</span>

#### Timepoints (Cluster permuation)

In [41]:
temporal_conds = ['Past', 'Futur', 'Distant Past', 'Distant Futur']
ch_list = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]

for ch_of_interest in ch_list:
    X_temporal = np.concatenate([
        epochs_ref[c].get_data(picks=[ch_of_interest])[:, 0, :]
        for c in temporal_conds
    ], axis=0)
    X_current = epochs_ref['Current'].get_data(picks=[ch_of_interest])[:, 0, :]


    df = X_temporal.shape[0] + X_current.shape[0] - 2
    threshold = t_dist.ppf(1 - 0.05 / 2, df=df)

    # ── Proper signed 2-sample test ──────────────────────────────────────────
    T_obs, clusters, cluster_pv, H0 = permutation_cluster_test(
        [X_temporal, X_current],
        stat_fun=signed_ttest,
        n_permutations=5000,    # increased for more precise p-values
        threshold=threshold,
        tail=0,
        n_jobs=-1,
        seed=42,
    )
    # ─────────────────────────────────────────────────────────────────────────

    # Split clusters by sign
    temporal_clusters = []
    current_clusters  = []
    for cluster, pval in zip(clusters, cluster_pv):
        if pval < 0.05:
            if T_obs[cluster[0]].mean() > 0:
                temporal_clusters.append((cluster, pval))
            else:
                current_clusters.append((cluster, pval))


    times = epochs_ref['Current'].times
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    ax = axes[0]
    for cond, color in zip(['Current'] + temporal_conds,
                        ['black', 'steelblue', 'cornflowerblue', 'tomato', 'salmon']):
        data = epochs_ref[cond].get_data(picks=[ch_of_interest])[:, 0, :]
        m    = data.mean(axis=0)
        s    = data.std(axis=0) / np.sqrt(len(data))
        ax.fill_between(times, m - s, m + s, alpha=0.12, color=color)
        ax.plot(times, m, lw=1.5, label=cond, color=color)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.axhline(0, color='k', lw=0.4, ls=':')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'Envelope: {ch_of_interest}, HGA band')

    for cluster, pval in temporal_clusters:
        ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
    for cluster, pval in current_clusters:
        ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')

    ax.legend(handles=[
        *[plt.Line2D([0],[0], color=c, lw=1.5, label=l)
        for c, l in zip(['black','steelblue','cornflowerblue','tomato','salmon'],
                        ['Current'] + temporal_conds)],
        Patch(color='green',  alpha=0.4, label='Temporal > Current (p<0.05)'),
        Patch(color='orange', alpha=0.4, label='Current > Temporal (p<0.05)'),
    ], fontsize=8, ncol=4)

    ax2 = axes[1]
    ax2.plot(times, T_obs, color='purple', lw=1.5)
    ax2.fill_between(times, 0, T_obs, where=T_obs > 0, alpha=0.3, color='green')
    ax2.fill_between(times, 0, T_obs, where=T_obs < 0, alpha=0.3, color='orange')
    ax2.axhline( threshold, color='gray', ls='--', lw=0.8, label=f'threshold ±{threshold:.2f}')
    ax2.axhline(-threshold, color='gray', ls='--', lw=0.8)
    ax2.axhline(0,          color='k',   lw=0.5)
    ax2.axvline(0,          color='k',   lw=0.8, ls='--')
    ax2.set_ylabel('T statistic\n+ = Temporal > Current\n− = Current > Temporal')
    ax2.set_xlabel('Time (s)')
    ax2.legend(fontsize=8)

    for cluster, pval in temporal_clusters:
        ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
        ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkgreen')
    for cluster, pval in current_clusters:
        ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')
        ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                -threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkorange')

    plt.tight_layout()
    plt.savefig(f"figures/{subject}/statistical_significance/{reference}/Enveloppe/Mixed_Temporal_vs_Current/HGA/cluster_perm_erp_{ch_of_interest}_HGA.png", dpi=150)
    plt.show()
    plt.close()

    print(f"\n{ch_of_interest}")
    print(f"  Temporal > Current (green):")
    for cluster, pval in temporal_clusters:
        print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
    if not temporal_clusters: print("    None")
    print(f"  Current > Temporal (orange):")
    for cluster, pval in current_clusters:
        print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
    if not current_clusters: print("    None")

stat_fun(H1): min=-2.5264797949889863 max=2.8478602201784167
Running initial clustering …
Found 13 clusters


  0%|          | Permuting : 0/4999 [00:00<?,       ?it/s]


TOD2-TOD3
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None
stat_fun(H1): min=-2.2102019835398283 max=2.906981736141291
Running initial clustering …
Found 6 clusters


  0%|          | Permuting : 0/4999 [00:00<?,       ?it/s]


TOD3-TOD4
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None
stat_fun(H1): min=-2.9152355695468084 max=2.422060084016172
Running initial clustering …
Found 11 clusters


...


HAD1-HAD2
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None
stat_fun(H1): min=-3.5976804476452404 max=1.7346782813812331
Running initial clustering …
Found 6 clusters


  0%|          | Permuting : 0/4999 [00:00<?,       ?it/s]


HAD2-HAD3
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None
stat_fun(H1): min=-3.1329937793160014 max=2.519216336051384
Running initial clustering …
Found 13 clusters


  0%|          | Permuting : 0/4999 [00:00<?,       ?it/s]


HAD3-HAD4
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None


Cluster-based permutation testing confirmed what we saw before, no statistical significant result/cluster for high gamma :(

#### Average between onset and answer (classification of the task)

Here we will not compare specific timepoints over an arbitrary period. Instead we will average the potential when the patient is deciding the condition of the shown task. This is more meaningful in terms of task-related response.

In [42]:
# Reepoch the data to include longer response periods
tmax_dur = 20
epochs_dur, epochs_dur_no_hga, band_mean_amps_dur, envelopes_dur = get_epochs_from_raw_filtered(raw_filtered, events, event_id, tmin, tmax_dur, subject=subject, reference=reference, pad=PAD)

threshold_ref = 8.5*1e7
late_tmin = 2
late_tmax = 7

epochs_dur_removed, good_mask = late_artifact_rejection(
    epochs_dur,
    threshold_uv=threshold_ref,   # can be stricter than full-epoch threshold
    late_tmin=late_tmin,       # adjust based on your expected response duration
    late_tmax=late_tmax
)

Not setting metadata
102 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 102 events and 45057 original time points ...
0 bad epochs dropped
Bipolar Referencing
sEEG channel type selected for re-referencing
Not setting metadata
102 matching events found
No baseline correction applied
0 projection items activated
Added the following bipolar channels:
TOD2-TOD3, TOD3-TOD4, PHD1-PHD2, PHD2-PHD3, PHD3-PHD4, PHD4-PHD5, HPD1-HPD2, HPD2-HPD3, HPD3-HPD4, HPD4-HPD5, HAD1-HAD2, HAD2-HAD3, HAD3-HAD4
Setting up band-pass filter from 70 - 80 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 70.00, 80.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    5.7s
[Parallel(n_jobs=9)]: Done 162 tasks      | elapsed:    6.0s
[Parallel(n_jobs=9)]: Done 1224 tasks      | elapsed:    8.0s
[Parallel(n_jobs=9)]: Done 1326 out of 1326 | elapsed:    8.2s finished


Setting up band-pass filter from 80 - 90 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 80.00, 90.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 855 tasks      | elapsed:    1.5s
[Parallel(n_jobs=9)]: Done 1305 tasks      | elapsed:    2.3s
[Parallel(n_jobs=9)]: Done 1326 out of 1326 | elapsed:    2.4s finished


Setting up band-pass filter from 90 - 1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 90.00, 100.00 Hz: -6.02, -6.02 dB



...

Setting up band-pass filter from 1.9e+02 - 2e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 190.00, 200.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 855 tasks      | elapsed:    1.6s
[Parallel(n_jobs=9)]: Done 1305 tasks      | elapsed:    2.4s
[Parallel(n_jobs=9)]: Done 1318 out of 1326 | elapsed:    2.4s remaining:    0.0s
[Parallel(n_jobs=9)]: Done 1326 out of 1326 | elapsed:    2.4s finished


Setting up band-pass filter from 2e+02 - 2.1e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 200.00, 210.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 855 tasks      | elapsed:    1.4s
[Parallel(n_jobs=9)]: Done 1305 tasks      | elapsed:    2.1s
[Parallel(n_jobs=9)]: Done 1326 out of 1326 | elapsed:    2.2s finished


Computed HGA envelope: (102, 13, 45057)  →  (epochs, channels, times)
Late window: t >= 2s
Threshold: 85000000.0 µV
Rejected 7 / 102 epochs (6.9%)
  Epoch 3: late artifact in ['PHD2-PHD3'] (late P2P=88513930µV, early P2P=82084174µV)
  Epoch 4: late artifact in ['HAD1-HAD2'] (late P2P=85208525µV, early P2P=111427720µV)
  Epoch 46: late artifact in ['HAD2-HAD3'] (late P2P=87439743µV, early P2P=90945913µV)
  Epoch 69: late artifact in ['PHD2-PHD3'] (late P2P=87586456µV, early P2P=92517360µV)
  Epoch 70: late artifact in ['PHD2-PHD3'] (late P2P=107104194µV, early P2P=114709589µV)
  Epoch 76: late artifact in ['PHD1-PHD2'] (late P2P=94727318µV, early P2P=89477201µV)
  Epoch 91: late artifact in ['HPD1-HPD2'] (late P2P=85824517µV, early P2P=94407511µV)


In [43]:
# Now we crop each trial from -1s to 0s + duration (of the specific trial) and get average between 0 and duration
epochs_ref_dur = []
late_answers = []
for i, ann in enumerate(epochs_dur_removed.annotations):
    if ep.annotations[i]["duration"] < 20:
        dur = ann['duration'] 
    else:
        dur = 20
        late_answers.append(i)

    ep = epochs_dur_removed[i].copy().crop(tmin=-1, tmax=dur) #
    ep._data = np.ascontiguousarray(ep._data)
    epochs_ref_dur.append(ep)

# Now we can compute the average amplitude in the 0–duration window for each trial
avg_amplitudes = []
condition_labels = []
for i,ep in enumerate(epochs_ref_dur):
    if i in late_answers:
        continue
    data = ep.get_data(picks='seeg')  # (1, n_ch, n_times)
    times = ep.times
    mask = (times >= 0) & (times <= ep.annotations[i]["duration"])  # from 0 to end of trial
    avg_amp = data[:, :, mask].mean(axis=-1)  # average over time
    avg_amplitudes.append(avg_amp)
    condition = ep.annotations[i]["description"]
    condition_labels.append(condition)
avg_amplitudes = np.array(avg_amplitudes)  # (n_trials, n_ch)

# Fusion distant and near conditions, and remove late answers
avg_amplitudes_on_time = []
condition_labels_on_time = []
for i in range(len(avg_amplitudes)):
    if i in late_answers:
        continue
    avg_amplitudes_on_time.append(avg_amplitudes[i])
    if condition_labels[i] == "Distant Futur" :
         condition_labels[i] = "Futur"
    if condition_labels[i] == "Distant Past" :
         condition_labels[i] = "Past"
    condition_labels_on_time.append(condition_labels[i])
condition_labels_on_time = np.array(condition_labels_on_time)

# Get baseline data (from -1s to 0s) for each trial
for i, ep in enumerate(epochs_ref_dur):
    if i in late_answers:
        continue
    baseline_mask = (epochs_ref_dur[i].times >= -1) & (epochs_ref_dur[i].times < 0)
    baseline_trial = epochs_ref_dur[i].get_data(picks='seeg')[:, :, baseline_mask]

    # Substract the average baseline to the epoch
    data = ep.get_data(picks='seeg')  
    baseline_mean = baseline_trial.mean(axis=-1) 
    data_db = [] 
    for ch in range(len(ep.ch_names)):
        data_norm = []
        data_ch = data[0,ch]
        #print(len(data_ch))
        for timepoint in range(len(data_ch)):
            data_norm.append(data_ch[timepoint] - baseline_mean[:,ch])
        data_db.append(data_norm)

    # Reshape to (n_epochs (1), nb_channels (13), window size (16k))
    data_db = np.permute_dims(data_db, (-1,0,-2))

    # Put normalized data back into the epochs object
    ep._data[:, :len(ep.ch_names), :] = data_db


In [44]:
def pval_label(p):
        if p < 0.001: return '***'
        if p < 0.01:  return '**'
        if p < 0.05:  return '*'
        return 'ns'

In [46]:
raise NotImplementedError("Stop here")

NotImplementedError: Stop here

In [54]:
for ch_of_interest in ch_list:
    ch_idx = epochs_ref.ch_names.index(ch_of_interest)

    unique_conditions = sorted(set(condition_labels_on_time))
    unique_conditions = ["Current", "Past", "Futur", "Futur+Past"]  # to keep a specific order

    means = []
    sems  = []
    ns    = []
    pvals_vs_current  = {}

    print(f"{ch_of_interest}")
    for cond in unique_conditions:
        if cond == "Futur+Past":
            indices = [i for i, c in enumerate(condition_labels_on_time) if c in ["Past", "Futur"]]
        else:
            indices = [i for i, c in enumerate(condition_labels_on_time) if c == cond]

        avg_amp_cond = np.array([avg_amplitudes_on_time[i] for i in indices])[:,:,ch_idx]  # (n_trials)

        # Store
        means.append(avg_amp_cond.mean()) # one average per trial
        sems.append(avg_amp_cond.std() / np.sqrt(len(indices)))
        ns.append(len(indices))

        if cond != "Current":
            # Perform 2 samples independant t-test against Current condition
            current_indices = [i for i, c in enumerate(condition_labels_on_time) if c == "Current"]
            avg_amp_current = np.array([avg_amplitudes_on_time[i] for i in current_indices])[:,:,ch_idx]

            t_stat, p_val = ttest_ind(avg_amp_cond, avg_amp_current, equal_var=False)
            if p_val < 0.05:
                print(f"{cond} vs Current: t={t_stat}, \033[91mp={p_val}\033[0m")
            pvals_vs_current[cond] = p_val


    means = np.array(means)
    sems  = np.array(sems)

    fig, ax = plt.subplots(figsize=(10, 5))

    colors = [cond_colors.get(c, 'gray') for c in unique_conditions]
    x      = np.arange(len(unique_conditions))

    ax.bar(x, means, yerr=sems, color=colors, alpha=0.7,
        capsize=5, error_kw=dict(lw=1.5), width=0.6)

    # Add n= above each bar
    for i, (m, n) in enumerate(zip(means, ns)):
        ax.text(i, m + sems[i] + np.abs(means).max() * 0.03,
                f'n={n}', ha='center', fontsize=9, color='gray')


    y_offset = np.abs(means).max() * 0.15   # vertical spacing for brackets
    y_top    = (means + sems).max() + y_offset

    cond_x = {c: i for i, c in enumerate(unique_conditions)}

    for cond in ["Past", "Futur", "Futur+Past"]:
        ci = cond_x[cond]

        # vs Current
        p_curr = pvals_vs_current.get(cond, None)[0]
        if p_curr is not None:
            x1, x2 = cond_x["Current"], ci
            y = y_top
            ax.plot([x1, x1, x2, x2], [y, y + y_offset*0.3, y + y_offset*0.3, y],
                    lw=1.0, color='black')
            ax.text((x1 + x2) , y + y_offset * 0.35,
                    f'curr: {p_curr:.3f}',
                    ha='center', fontsize=8,
                    color='red' if p_curr < 0.05 else 'gray')

    ax.axhline(0, color='k', lw=0.8, ls=':')
    ax.set_xticks(x)
    ax.set_xticklabels(unique_conditions, rotation=20, ha='right', fontsize=10)
    ax.set_title(f'Average amplitude per condition — {ch_of_interest}')
    ax.set_xlabel('Condition')
    ax.set_ylabel('Average amplitude (V)')
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"figures/{subject}/statistical_significance/{reference}/Average/average_amplitude_per_condition_{ch_of_interest}.png",
                dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

TOD2-TOD3
Futur vs Current: t=[2.59101604], p=[0.01585624]
Futur+Past vs Current: t=[2.53713585], p=[0.01602633]
TOD3-TOD4
PHD1-PHD2
PHD2-PHD3
PHD3-PHD4
PHD4-PHD5
HPD1-HPD2
HPD2-HPD3
HPD3-HPD4
HPD4-HPD5
HAD1-HAD2
HAD2-HAD3
HAD3-HAD4


In [ ]:
# Plot histogram of epoch durations
durations = epochs_ref.annotations.duration
plt.figure(figsize=(8, 4))
plt.hist(durations, bins=20, color='skyblue', edgecolor='k')
plt.xlabel('Epoch duration (s)')
plt.ylabel('Number of epochs')
plt.title('Distribution of epoch durations')
plt.tight_layout()
plt.savefig(f"figures/{subject}/epoch_durations_histogram.png", dpi=300)
plt.show()
plt.close()

/tmp/ipykernel_1198361/422706163.py:3: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(8, 4))


In [ ]:
# Stop here
raise NotImplementedError("Stop here for now")

### <span style="color:orange">Epochs computing</span>

Compute frequency bands from epochs for statistical analysis (same as before but I include all conditions):

In [ ]:
bands = {
    "theta":     (4,   8),
    "alpha":     (8,  12),
    "beta":      (12, 30),
    "low_gamma": (30, 80),
    "high_gamma":(80, 200),
}
cond_list_stats = ["Current", "Never", "Always", "Past", "Futur", "Distant Past", "Distant Futur"]  
epochs_bands_stats = {} # the difference with epoch_bands is that stat has all conditions
for cond in cond_list_stats:
    for band_name, (fmin, fmax) in bands.items():
        print(f"Processing {cond} — {band_name} ({fmin}-{fmax} Hz)")
        # copy the referenced epochs (before any filtering)
        ep = epochs_ref[cond].copy()
        ep._data = np.ascontiguousarray(ep._data) # wtffffff
        
        # bandpass to the band of interest
        ep.filter(fmin, fmax, picks='seeg', method="iir", iir_params=None, n_jobs=9)
        
        # extract envelope
        ep.apply_hilbert(envelope=True)
        ep.filter(None, fmin/2, picks='seeg', method="iir", iir_params=None, n_jobs=9)
        
        # baseline normalization
        data = ep.get_data(picks='seeg')
        baseline_mask = ep.times < 0
        baseline_mean = data[:, :, baseline_mask].mean(axis=-1, keepdims=True)
        ep._data[:, :len(ep.ch_names), :] = data - baseline_mean

        epochs_bands_stats[(band_name, cond)] = ep.crop(tmin=tmin, tmax=tmax)

Processing Current — theta (4-8 Hz)
Setting up band-pass filter from 4 - 8 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 4.00, 8.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 162 tasks      | elapsed:    0.2s


Setting up low-pass filter at 2 Hz

IIR filter parameters
---------------------
Butterworth lowpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 2.00 Hz: -6.02 dB



[Parallel(n_jobs=9)]: Done 221 out of 221 | elapsed:    0.2s finished
[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 162 tasks      | elapsed:    0.2s


Processing Current — alpha (8-12 Hz)
Setting up band-pass filter from 8 - 12 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 8.00, 12.00 Hz: -6.02, -6.02 dB



...

[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 156 out of 156 | elapsed:    0.1s finished


Processing Distant Futur — high_gamma (80-200 Hz)
Setting up band-pass filter from 80 - 2e+02 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 80.00, 200.00 Hz: -6.02, -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 156 out of 156 | elapsed:    0.1s finished


Setting up low-pass filter at 40 Hz

IIR filter parameters
---------------------
Butterworth lowpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 8 (effective, after forward-backward)
- Cutoff at 40.00 Hz: -6.02 dB



[Parallel(n_jobs=9)]: Using backend LokyBackend with 9 concurrent workers.
[Parallel(n_jobs=9)]: Done   9 tasks      | elapsed:    0.0s
[Parallel(n_jobs=9)]: Done 156 out of 156 | elapsed:    0.2s finished


### <span style="color:orange">T-test, Mann-Whitney, Cohen d-value</span>

These tests consider the average value of the potential, ignoring time-related differences between responses. The next section will investigate time-related statistical analysis.

#### Heatmaps, delete?

In [ ]:
bands     = ['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']
cond_keys = ['Past', 'Futur', 'Distant Past', 'Distant Futur']   # each compared against Current

tmin_win, tmax_win = 0.0, 3.0   # adjust after looking at the plots

results = []

for band in bands:
    ep_current = epochs_bands_stats[(band, 'Current')]
    times      = ep_current.times
    tmin_idx   = ep_current.time_as_index(tmin_win)[0]
    tmax_idx   = ep_current.time_as_index(tmax_win)[0]
    ch_names   = ep_current.ch_names

    # Extract amplitude envelope for Current once per band
    data_curr = ep_current.get_data()                          # (n_trials, n_ch, n_t)
    amp_curr  = np.abs(hilbert(data_curr, axis=-1))

    for cond in cond_keys:
        ep_cond  = epochs_bands_stats[(band, cond)]
        data_cond = ep_cond.get_data()
        amp_cond  = np.abs(hilbert(data_cond, axis=-1))

        for ch_idx, ch_name in enumerate(ch_names):
            a = amp_cond[:, ch_idx, tmin_idx:tmax_idx].mean(axis=1)   # scalar per trial
            b = amp_curr[:, ch_idx, tmin_idx:tmax_idx].mean(axis=1)

            t,   p_t  = ttest_ind(a, b)
            _, p_mw   = mannwhitneyu(a, b, alternative='two-sided')

            results.append(dict(
                band=band, condition=cond, channel=ch_name,
                n_cond=len(a), n_current=len(b),
                t=t, p_ttest=p_t, p_mannwhitney=p_mw,
                mean_cond=a.mean(), mean_current=b.mean(),
                cohen_d=(a.mean() - b.mean()) / np.sqrt(
                    ((len(a)-1)*a.std()**2 + (len(b)-1)*b.std()**2) /
                    (len(a) + len(b) - 2)
                )
            ))

df_results = pd.DataFrame(results)

# FDR correction across all tests within each band×condition pair
df_results['p_fdr'] = np.nan
for (band, cond), grp in df_results.groupby(['band', 'condition']):
    reject, p_corr = fdr_correction(grp['p_ttest'].values, alpha=0.05)
    df_results.loc[grp.index, 'p_fdr']   = p_corr
    df_results.loc[grp.index, 'sig_fdr'] = reject

# ── Summary of significant results ───────────────────────────────────────────
sig = df_results[df_results['sig_fdr'] == True]
print(f"\n{len(sig)} significant channel×band×condition combinations (FDR p<0.05):\n")
print(sig[['band','condition','channel','t','p_ttest','p_fdr','cohen_d','n_cond','n_current']].to_string(index=False))

# ── Full table sorted by p-value ──────────────────────────────────────────────
print("\nFull results (sorted by p_ttest):")
print(df_results.sort_values('p_ttest')[
    ['band','condition','channel','t','p_ttest','p_mannwhitney','p_fdr','cohen_d']
].to_string(index=False))


0 significant channel×band×condition combinations (FDR p<0.05):

Empty DataFrame
Columns: [band, condition, channel, t, p_ttest, p_fdr, cohen_d, n_cond, n_current]
Index: []

Full results (sorted by p_ttest):
      band     condition   channel         t  p_ttest  p_mannwhitney    p_fdr   cohen_d
     theta  Distant Past PHD3-PHD4  2.806742 0.008852       0.005802 0.115079  1.046931
     alpha Distant Futur HPD1-HPD2  2.430541 0.021998       0.013988 0.285969  0.953695
high_gamma  Distant Past HAD3-HAD4  2.395165 0.023293       0.037165 0.302809  0.895140
high_gamma          Past PHD4-PHD5 -2.396011 0.024063       0.027043 0.312816 -0.961374
high_gamma Distant Futur PHD4-PHD5 -2.312225 0.028628       0.031744 0.372165 -0.900959
     alpha          Past TOD3-TOD4 -2.164010 0.039832       0.048189 0.517820 -0.866235
     alpha         Futur TOD2-TOD3  2.182847 0.040544       0.155091 0.527070  1.105626
      beta          Past HPD1-HPD2 -2.155305 0.040574       0.090369 0.527461 -0.86511

In [ ]:
# Visualization Heatmap

# Pivot cohen_d into a (band × channel) matrix per condition
fig, axes = plt.subplots(1, len(cond_keys), figsize=(14, 5), sharey=True)

for ax, cond in zip(axes, cond_keys):
    pivot = df_results[df_results['condition'] == cond].pivot(
        index='band', columns='channel', values='cohen_d'
    ).reindex(bands)

    p_pivot = df_results[df_results['condition'] == cond].pivot(
        index='band', columns='channel', values='p_fdr'
    ).reindex(bands)

    im = ax.imshow(pivot.values, aspect='auto', cmap='RdBu_r',
                   norm=mcolors.TwoSlopeNorm(vcenter=0, vmin=-1.5, vmax=1.5))

    # Mark significant cells
    for r in range(pivot.shape[0]):
        for c in range(pivot.shape[1]):
            if p_pivot.values[r, c] < 0.05:
                ax.text(c, r, '*', ha='center', va='center', fontsize=14, color='k')

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(bands)))
    ax.set_yticklabels(bands)
    ax.set_title(f'{cond} vs Current')

plt.colorbar(im, ax=axes[-1], label="Cohen's d")
plt.suptitle("Effect size (Cohen's d): condition vs Current\n* = FDR p<0.05")
plt.tight_layout()
plt.show()

plt.savefig(f"figures/{subject}/statistical_significance/{reference}/{subject}_{reference}_effect_size_heatmap_separate_cond.png", dpi=300)
plt.close()

Past and future together vs current:

In [ ]:
bands     = ['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']
tmin_win, tmax_win = 0.0, 3.0

results = []

for band in bands:
    ep_current = epochs_bands_stats[(band, 'Current')]
    times      = ep_current.times
    tmin_idx   = ep_current.time_as_index(tmin_win)[0]
    tmax_idx   = ep_current.time_as_index(tmax_win)[0]
    ch_names   = ep_current.ch_names

    amp_curr = ep_current.get_data(picks='seeg')  # already envelope from pipeline

    # Pool all temporal conditions
    temporal_conds = ['Past', 'Futur', 'Distant Past', 'Distant Futur']
    amp_temporal = np.concatenate([
        epochs_bands_stats[(band, cond)].get_data(picks='seeg')
        for cond in temporal_conds
    ], axis=0)

    print(f"{band}: temporal n={amp_temporal.shape[0]}, current n={amp_curr.shape[0]}")

    for ch_idx, ch_name in enumerate(ch_names):
        a = amp_temporal[:, ch_idx, tmin_idx:tmax_idx].mean(axis=1)
        b = amp_curr[:,    ch_idx, tmin_idx:tmax_idx].mean(axis=1)

        t,  p_t  = ttest_ind(a, b)
        _,  p_mw = mannwhitneyu(a, b, alternative='two-sided')

        n_a, n_b = len(a), len(b)
        pooled_std = np.sqrt(
            ((n_a - 1) * a.std()**2 + (n_b - 1) * b.std()**2) / (n_a + n_b - 2)
        )
        d = (a.mean() - b.mean()) / (pooled_std + 1e-12)

        results.append(dict(
            band=band, condition='Temporal', channel=ch_name,
            n_temporal=n_a, n_current=n_b,
            t=t, p_ttest=p_t, p_mannwhitney=p_mw,
            mean_temporal=a.mean(), mean_current=b.mean(),
            cohen_d=d
        ))

df_pooled = pd.DataFrame(results)

# FDR correction per band
df_pooled['p_fdr']   = np.nan
df_pooled['sig_fdr'] = False
for band, grp in df_pooled.groupby('band'):
    reject, p_corr = fdr_correction(grp['p_ttest'].values, alpha=0.05)
    df_pooled.loc[grp.index, 'p_fdr']   = p_corr
    df_pooled.loc[grp.index, 'sig_fdr'] = reject

sig = df_pooled[df_pooled['sig_fdr']]
print(f"\n{len(sig)} significant results (FDR p<0.05)\n")
print(sig[['band','channel','t','p_ttest','p_fdr','cohen_d','n_temporal','n_current']].to_string(index=False))

print("\nFull results sorted by p_ttest:")
print(df_pooled.sort_values('p_ttest')[
    ['band','channel','t','p_ttest','p_mannwhitney','p_fdr','cohen_d','n_temporal','n_current']
].to_string(index=False))

theta: temporal n=43, current n=17
alpha: temporal n=43, current n=17
beta: temporal n=43, current n=17
low_gamma: temporal n=43, current n=17
high_gamma: temporal n=43, current n=17

0 significant results (FDR p<0.05)

Empty DataFrame
Columns: [band, channel, t, p_ttest, p_fdr, cohen_d, n_temporal, n_current]
Index: []

Full results sorted by p_ttest:
      band   channel         t  p_ttest  p_mannwhitney    p_fdr   cohen_d  n_temporal  n_current
      beta HAD3-HAD4 -2.704555 0.008963       0.012649 0.116513 -0.785663          43         17
      beta HPD2-HPD3 -2.024650 0.047516       0.144286 0.302212 -0.589669          43         17
     alpha PHD2-PHD3 -1.929538 0.058561       0.054941 0.550620 -0.561819          43         17
      beta HPD1-HPD2 -1.847741 0.069741       0.052898 0.302212 -0.540140          43         17
     theta PHD2-PHD3 -1.828887 0.072559       0.079208 0.655893 -0.532732          43         17
high_gamma TOD2-TOD3 -1.682948 0.097761       0.054941 0.914158

In [ ]:
bands_order = ['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']

pivot_d = df_pooled.pivot(index='band', columns='channel', values='cohen_d').reindex(bands_order)
pivot_p = df_pooled.pivot(index='band', columns='channel', values='p_fdr').reindex(bands_order)

fig, ax = plt.subplots(figsize=(14, 4))

im = ax.imshow(
    pivot_d.values,
    aspect='auto',
    cmap='RdBu_r',
    norm=mcolors.TwoSlopeNorm(vcenter=0, vmin=-1.5, vmax=1.5)
)

# Mark significant cells
for r in range(pivot_d.shape[0]):
    for c in range(pivot_d.shape[1]):
        if pivot_p.values[r, c] < 0.05:
            ax.text(c, r, '*', ha='center', va='center', fontsize=14, color='k')

ax.set_xticks(range(len(pivot_d.columns)))
ax.set_xticklabels(pivot_d.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(bands_order)))
ax.set_yticklabels(bands_order)
ax.set_title('Temporal (Past + Futur + Distant Past + Distant Futur) vs Current')

plt.colorbar(im, ax=ax, label="Cohen's d")
plt.suptitle("Effect size (Cohen's d): Temporal vs Current\n* = FDR p<0.05")
plt.tight_layout()

output_path = f"figures/{subject}/statistical_significance/{reference}/{subject}_{reference}_effect_size_heatmap_temporal.png"
plt.savefig(output_path, dpi=300)
plt.show()
print(f"Saved to {output_path}")
plt.close()

Saved to figures/02/statistical_significance/bipolar/02_bipolar_effect_size_heatmap_temporal.png


Past (Distant + near) and Futur (Distant + near) against current:

In [ ]:
results_split = []

for band in bands:
    ep_current = epochs_bands_stats[(band, 'Current')]
    times      = ep_current.times
    tmin_idx   = ep_current.time_as_index(tmin_win)[0]
    tmax_idx   = ep_current.time_as_index(tmax_win)[0]
    ch_names   = ep_current.ch_names

    amp_curr = ep_current.get_data(picks='seeg')

    groups = {
        'Past':  ['Past', 'Distant Past'],
        'Futur': ['Futur', 'Distant Futur'],
    }

    for group_name, cond_list in groups.items():
        amp_group = np.concatenate([
            epochs_bands_stats[(band, cond)].get_data(picks='seeg')
            for cond in cond_list
        ], axis=0)

        print(f"{band} | {group_name}: n={amp_group.shape[0]}, current n={amp_curr.shape[0]}")

        for ch_idx, ch_name in enumerate(ch_names):
            a = amp_group[:, ch_idx, tmin_idx:tmax_idx].mean(axis=1)
            b = amp_curr[:,  ch_idx, tmin_idx:tmax_idx].mean(axis=1)

            t,  p_t  = ttest_ind(a, b)
            _,  p_mw = mannwhitneyu(a, b, alternative='two-sided')

            n_a, n_b = len(a), len(b)
            pooled_std = np.sqrt(
                ((n_a - 1) * a.std()**2 + (n_b - 1) * b.std()**2) / (n_a + n_b - 2)
            )
            d = (a.mean() - b.mean()) / (pooled_std + 1e-12)

            results_split.append(dict(
                band=band, condition=group_name, channel=ch_name,
                n_group=n_a, n_current=n_b,
                t=t, p_ttest=p_t, p_mannwhitney=p_mw,
                mean_group=a.mean(), mean_current=b.mean(),
                cohen_d=d
            ))

df_split = pd.DataFrame(results_split)

# FDR correction per band × condition
df_split['p_fdr']   = np.nan
df_split['sig_fdr'] = False
for (band, cond), grp in df_split.groupby(['band', 'condition']):
    reject, p_corr = fdr_correction(grp['p_ttest'].values, alpha=0.05)
    df_split.loc[grp.index, 'p_fdr']   = p_corr
    df_split.loc[grp.index, 'sig_fdr'] = reject

sig = df_split[df_split['sig_fdr']]
print(f"\n{len(sig)} significant results (FDR p<0.05)\n")

print("\nFull results sorted by p_ttest:")
print(df_split.sort_values('p_ttest')[
    ['band', 'condition', 'channel', 't', 'p_ttest', 'p_mannwhitney', 'p_fdr', 'cohen_d', 'n_group', 'n_current']
].to_string(index=False))

theta | Past: n=25, current n=17
theta | Futur: n=18, current n=17
alpha | Past: n=25, current n=17
alpha | Futur: n=18, current n=17
beta | Past: n=25, current n=17
beta | Futur: n=18, current n=17
low_gamma | Past: n=25, current n=17
low_gamma | Futur: n=18, current n=17
high_gamma | Past: n=25, current n=17
high_gamma | Futur: n=18, current n=17

1 significant results (FDR p<0.05)


Full results sorted by p_ttest:
      band condition   channel         t  p_ttest  p_mannwhitney    p_fdr   cohen_d  n_group  n_current
      beta     Futur HAD3-HAD4 -3.513337 0.001307       0.003490 0.016991 -1.223242       18         17
      beta     Futur HPD1-HPD2 -2.392327 0.022594       0.015272 0.146859 -0.833552       18         17
      beta      Past HPD2-HPD3 -2.289211 0.027423       0.106448 0.356498 -0.737550       25         17
     alpha     Futur PHD4-PHD5  2.067641 0.046588       0.062213 0.395097  0.720047       18         17
     alpha      Past PHD3-PHD4 -2.044442 0.047530       0.0

In [ ]:
bands_order = ['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']
groups      = ['Past', 'Futur']

fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=True)

for ax, group in zip(axes, groups):
    pivot_d = df_split[df_split['condition'] == group].pivot(
        index='band', columns='channel', values='cohen_d'
    ).reindex(bands_order)

    pivot_p = df_split[df_split['condition'] == group].pivot(
        index='band', columns='channel', values='p_fdr'
    ).reindex(bands_order)

    im = ax.imshow(
        pivot_d.values,
        aspect='auto',
        cmap='RdBu_r',
        norm=mcolors.TwoSlopeNorm(vcenter=0, vmin=-1.5, vmax=1.5)
    )

    for r in range(pivot_d.shape[0]):
        for c in range(pivot_d.shape[1]):
            if pivot_p.values[r, c] < 0.05:
                ax.text(c, r, '*', ha='center', va='center', fontsize=14, color='k')

    ax.set_xticks(range(len(pivot_d.columns)))
    ax.set_xticklabels(pivot_d.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(bands_order)))
    ax.set_yticklabels(bands_order)
    ax.set_title(f'{group} (near + distant) vs Current')

plt.colorbar(im, ax=axes[-1], label="Cohen's d")
plt.suptitle("Effect size (Cohen's d)\n* = FDR p<0.05")
plt.tight_layout()

output_path = f"figures/{subject}/statistical_significance/{reference}/{subject}_{reference}_past_futur_heatmap.png"
plt.savefig(output_path, dpi=300)
plt.show()
print(f"Saved to {output_path}")
plt.close()

Saved to figures/02/statistical_significance/bipolar/02_bipolar_past_futur_heatmap.png


### <span style="color:orange">Time-related statistical analysis</span>

#### Temporal vs Current: ERP difference

Signed T-curve: Mixed temp, All past, All future

In [ ]:
temporal_conds = ['Past', 'Futur', 'Distant Past', 'Distant Futur']
ch_list = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]

for ch_of_interest in ch_list:
    X_temporal = np.concatenate([
        epochs_ref[c].get_data(picks=[ch_of_interest])[:, 0, :]
        for c in temporal_conds
    ], axis=0)
    X_current = epochs_ref['Current'].get_data(picks=[ch_of_interest])[:, 0, :]


    df        = X_temporal.shape[0] + X_current.shape[0] - 2
    threshold = t_dist.ppf(1 - 0.05 / 2, df=df)

    # ── Proper signed 2-sample test ──────────────────────────────────────────
    T_obs, clusters, cluster_pv, H0 = permutation_cluster_test(
        [X_temporal, X_current],
        stat_fun=signed_ttest,
        n_permutations=5000,    # increased for more precise p-values
        threshold=threshold,
        tail=0,
        n_jobs=-1,
        seed=42,
    )
    # ─────────────────────────────────────────────────────────────────────────

    # Split clusters by sign
    temporal_clusters = []
    current_clusters  = []
    for cluster, pval in zip(clusters, cluster_pv):
        if pval < 0.05:
            if T_obs[cluster[0]].mean() > 0:
                temporal_clusters.append((cluster, pval))
            else:
                current_clusters.append((cluster, pval))

  
    times = epochs_ref['Current'].times
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    ax = axes[0]
    for cond, color in zip(['Current'] + temporal_conds,
                           ['black', 'steelblue', 'cornflowerblue', 'tomato', 'salmon']):
        data = epochs_ref[cond].get_data(picks=[ch_of_interest])[:, 0, :]
        m    = data.mean(axis=0)
        s    = data.std(axis=0) / np.sqrt(len(data))
        ax.fill_between(times, m - s, m + s, alpha=0.12, color=color)
        ax.plot(times, m, lw=1.5, label=cond, color=color)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.axhline(0, color='k', lw=0.4, ls=':')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'ERP: {ch_of_interest}')

    for cluster, pval in temporal_clusters:
        ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
    for cluster, pval in current_clusters:
        ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')

    ax.legend(handles=[
        *[plt.Line2D([0],[0], color=c, lw=1.5, label=l)
          for c, l in zip(['black','steelblue','cornflowerblue','tomato','salmon'],
                          ['Current'] + temporal_conds)],
        Patch(color='green',  alpha=0.4, label='Temporal > Current (p<0.05)'),
        Patch(color='orange', alpha=0.4, label='Current > Temporal (p<0.05)'),
    ], fontsize=8, ncol=4)

    ax2 = axes[1]
    ax2.plot(times, T_obs, color='purple', lw=1.5)
    ax2.fill_between(times, 0, T_obs, where=T_obs > 0, alpha=0.3, color='green')
    ax2.fill_between(times, 0, T_obs, where=T_obs < 0, alpha=0.3, color='orange')
    ax2.axhline( threshold, color='gray', ls='--', lw=0.8, label=f'threshold ±{threshold:.2f}')
    ax2.axhline(-threshold, color='gray', ls='--', lw=0.8)
    ax2.axhline(0,          color='k',   lw=0.5)
    ax2.axvline(0,          color='k',   lw=0.8, ls='--')
    ax2.set_ylabel('T statistic\n+ = Temporal > Current\n− = Current > Temporal')
    ax2.set_xlabel('Time (s)')
    ax2.legend(fontsize=8)

    for cluster, pval in temporal_clusters:
        ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
        ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                 threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkgreen')
    for cluster, pval in current_clusters:
        ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')
        ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                 -threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkorange')

    plt.tight_layout()
    plt.savefig(f"figures/{subject}/statistical_significance/{reference}/ERP/Mixed_Temporal_vs_Current/cluster_perm_erp_{ch_of_interest}.png", dpi=150)
    plt.show()
    plt.close()

    print(f"\n{ch_of_interest}")
    print(f"  Temporal > Current (green):")
    for cluster, pval in temporal_clusters:
        print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
    if not temporal_clusters: print("    None")
    print(f"  Current > Temporal (orange):")
    for cluster, pval in current_clusters:
        print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
    if not current_clusters: print("    None")

In [ ]:
temporal_conds = ['Past', 'Distant Past']
ch_list = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]

for ch_of_interest in ch_list:
    X_temporal = np.concatenate([
        epochs_ref[c].get_data(picks=[ch_of_interest])[:, 0, :]
        for c in temporal_conds
    ], axis=0)
    X_current = epochs_ref['Current'].get_data(picks=[ch_of_interest])[:, 0, :]


    df        = X_temporal.shape[0] + X_current.shape[0] - 2
    threshold = t_dist.ppf(1 - 0.05 / 2, df=df)

    # ── Proper signed 2-sample test ──────────────────────────────────────────
    T_obs, clusters, cluster_pv, H0 = permutation_cluster_test(
        [X_temporal, X_current],
        stat_fun=signed_ttest,
        n_permutations=5000,    # increased for more precise p-values
        threshold=threshold,
        tail=0,
        n_jobs=-1,
        seed=42,
    )
    # ─────────────────────────────────────────────────────────────────────────

    # Split clusters by sign
    temporal_clusters = []
    current_clusters  = []
    for cluster, pval in zip(clusters, cluster_pv):
        if pval < 0.05:
            if T_obs[cluster[0]].mean() > 0:
                temporal_clusters.append((cluster, pval))
            else:
                current_clusters.append((cluster, pval))

  
    times = epochs_ref['Current'].times
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    ax = axes[0]
    for cond, color in zip(['Current'] + temporal_conds,
                           ['black', 'steelblue', 'cornflowerblue', 'tomato', 'salmon']):
        data = epochs_ref[cond].get_data(picks=[ch_of_interest])[:, 0, :]
        m    = data.mean(axis=0)
        s    = data.std(axis=0) / np.sqrt(len(data))
        ax.fill_between(times, m - s, m + s, alpha=0.12, color=color)
        ax.plot(times, m, lw=1.5, label=cond, color=color)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.axhline(0, color='k', lw=0.4, ls=':')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'ERP: {ch_of_interest}')

    for cluster, pval in temporal_clusters:
        ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
    for cluster, pval in current_clusters:
        ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')

    ax.legend(handles=[
        *[plt.Line2D([0],[0], color=c, lw=1.5, label=l)
          for c, l in zip(['black','steelblue','cornflowerblue','tomato','salmon'],
                          ['Current'] + temporal_conds)],
        Patch(color='green',  alpha=0.4, label='Temporal > Current (p<0.05)'),
        Patch(color='orange', alpha=0.4, label='Current > Temporal (p<0.05)'),
    ], fontsize=8, ncol=4)

    ax2 = axes[1]
    ax2.plot(times, T_obs, color='purple', lw=1.5)
    ax2.fill_between(times, 0, T_obs, where=T_obs > 0, alpha=0.3, color='green')
    ax2.fill_between(times, 0, T_obs, where=T_obs < 0, alpha=0.3, color='orange')
    ax2.axhline( threshold, color='gray', ls='--', lw=0.8, label=f'threshold ±{threshold:.2f}')
    ax2.axhline(-threshold, color='gray', ls='--', lw=0.8)
    ax2.axhline(0,          color='k',   lw=0.5)
    ax2.axvline(0,          color='k',   lw=0.8, ls='--')
    ax2.set_ylabel('T statistic\n+ = Temporal > Current\n− = Current > Temporal')
    ax2.set_xlabel('Time (s)')
    ax2.legend(fontsize=8)

    for cluster, pval in temporal_clusters:
        ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
        ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                 threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkgreen')
    for cluster, pval in current_clusters:
        ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')
        ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                 -threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkorange')

    plt.tight_layout()
    plt.savefig(f"figures/{subject}/statistical_significance/{reference}/ERP/All_Past_vs_Current/cluster_perm_erp_{ch_of_interest}.png", dpi=150)
    plt.show()
    plt.close()

    print(f"\n{ch_of_interest}")
    print(f"  Temporal > Current (green):")
    for cluster, pval in temporal_clusters:
        print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
    if not temporal_clusters: print("    None")
    print(f"  Current > Temporal (orange):")
    for cluster, pval in current_clusters:
        print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
    if not current_clusters: print("    None")

In [ ]:
temporal_conds = ['Futur', 'Distant Futur']
ch_list = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]

for ch_of_interest in ch_list:
    X_temporal = np.concatenate([
        epochs_ref[c].get_data(picks=[ch_of_interest])[:, 0, :]
        for c in temporal_conds
    ], axis=0)
    X_current = epochs_ref['Current'].get_data(picks=[ch_of_interest])[:, 0, :]


    df        = X_temporal.shape[0] + X_current.shape[0] - 2
    threshold = t_dist.ppf(1 - 0.05 / 2, df=df)

    # ── Proper signed 2-sample test ──────────────────────────────────────────
    T_obs, clusters, cluster_pv, H0 = permutation_cluster_test(
        [X_temporal, X_current],
        stat_fun=signed_ttest,
        n_permutations=5000,    # increased for more precise p-values
        threshold=threshold,
        tail=0,
        n_jobs=-1,
        seed=42,
    )
    # ─────────────────────────────────────────────────────────────────────────

    # Split clusters by sign
    temporal_clusters = []
    current_clusters  = []
    for cluster, pval in zip(clusters, cluster_pv):
        if pval < 0.05:
            if T_obs[cluster[0]].mean() > 0:
                temporal_clusters.append((cluster, pval))
            else:
                current_clusters.append((cluster, pval))

  
    times = epochs_ref['Current'].times
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    ax = axes[0]
    for cond, color in zip(['Current'] + temporal_conds,
                           ['black', 'steelblue', 'cornflowerblue', 'tomato', 'salmon']):
        data = epochs_ref[cond].get_data(picks=[ch_of_interest])[:, 0, :]
        m    = data.mean(axis=0)
        s    = data.std(axis=0) / np.sqrt(len(data))
        ax.fill_between(times, m - s, m + s, alpha=0.12, color=color)
        ax.plot(times, m, lw=1.5, label=cond, color=color)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.axhline(0, color='k', lw=0.4, ls=':')
    ax.set_ylabel('Amplitude (µV)')
    ax.set_title(f'ERP: {ch_of_interest}')

    for cluster, pval in temporal_clusters:
        ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
    for cluster, pval in current_clusters:
        ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')

    ax.legend(handles=[
        *[plt.Line2D([0],[0], color=c, lw=1.5, label=l)
          for c, l in zip(['black','steelblue','cornflowerblue','tomato','salmon'],
                          ['Current'] + temporal_conds)],
        Patch(color='green',  alpha=0.4, label='Temporal > Current (p<0.05)'),
        Patch(color='orange', alpha=0.4, label='Current > Temporal (p<0.05)'),
    ], fontsize=8, ncol=4)

    ax2 = axes[1]
    ax2.plot(times, T_obs, color='purple', lw=1.5)
    ax2.fill_between(times, 0, T_obs, where=T_obs > 0, alpha=0.3, color='green')
    ax2.fill_between(times, 0, T_obs, where=T_obs < 0, alpha=0.3, color='orange')
    ax2.axhline( threshold, color='gray', ls='--', lw=0.8, label=f'threshold ±{threshold:.2f}')
    ax2.axhline(-threshold, color='gray', ls='--', lw=0.8)
    ax2.axhline(0,          color='k',   lw=0.5)
    ax2.axvline(0,          color='k',   lw=0.8, ls='--')
    ax2.set_ylabel('T statistic\n+ = Temporal > Current\n− = Current > Temporal')
    ax2.set_xlabel('Time (s)')
    ax2.legend(fontsize=8)

    for cluster, pval in temporal_clusters:
        ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
        ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                 threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkgreen')
    for cluster, pval in current_clusters:
        ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')
        ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                 -threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkorange')

    plt.tight_layout()
    plt.savefig(f"figures/{subject}/statistical_significance/{reference}/ERP/All_Future_vs_Current/cluster_perm_erp_{ch_of_interest}.png", dpi=150)
    plt.show()
    plt.close()

    print(f"\n{ch_of_interest}")
    print(f"  Temporal > Current (green):")
    for cluster, pval in temporal_clusters:
        print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
    if not temporal_clusters: print("    None")
    print(f"  Current > Temporal (orange):")
    for cluster, pval in current_clusters:
        print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
    if not current_clusters: print("    None")

#### Temporal vs Current: Enveloppe difference

In [ ]:
bands_order

['theta', 'alpha', 'beta', 'low_gamma', 'high_gamma']

In [ ]:
temporal_conds = ['Past', 'Futur', 'Distant Past', 'Distant Futur']
ch_list = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]
for band in bands_order:
    for ch_of_interest in ch_list:
        X_temporal = np.concatenate([
            epochs_bands_stats[(band, c)].get_data(picks=[ch_of_interest])[:, 0, :]
            for c in temporal_conds
        ], axis=0)
        X_current = epochs_bands_stats[(band,'Current')].get_data(picks=[ch_of_interest])[:, 0, :]


        df        = X_temporal.shape[0] + X_current.shape[0] - 2
        threshold = t_dist.ppf(1 - 0.05 / 2, df=df)

        # ── Proper signed 2-sample test ──────────────────────────────────────────
        T_obs, clusters, cluster_pv, H0 = permutation_cluster_test(
            [X_temporal, X_current],
            stat_fun=signed_ttest,
            n_permutations=5000,    # increased for more precise p-values
            threshold=threshold,
            tail=0,
            n_jobs=-1,
            seed=42,
        )
        # ─────────────────────────────────────────────────────────────────────────

        # Split clusters by sign
        temporal_clusters = []
        current_clusters  = []
        for cluster, pval in zip(clusters, cluster_pv):
            if pval < 0.05:
                if T_obs[cluster[0]].mean() > 0:
                    temporal_clusters.append((cluster, pval))
                else:
                    current_clusters.append((cluster, pval))

    
        times = epochs_bands_stats[(band, 'Current')].times
        fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

        ax = axes[0]
        for cond, color in zip(['Current'] + temporal_conds,
                            ['black', 'steelblue', 'cornflowerblue', 'tomato', 'salmon']):
            data = epochs_bands_stats[(band, cond)].get_data(picks=[ch_of_interest])[:, 0, :]
            m    = data.mean(axis=0)
            s    = data.std(axis=0) / np.sqrt(len(data))
            ax.fill_between(times, m - s, m + s, alpha=0.12, color=color)
            ax.plot(times, m, lw=1.5, label=cond, color=color)
        ax.axvline(0, color='k', lw=0.8, ls='--')
        ax.axhline(0, color='k', lw=0.4, ls=':')
        ax.set_ylabel('Amplitude (µV)')
        ax.set_title(f'Envelope: {ch_of_interest}, {band}')

        for cluster, pval in temporal_clusters:
            ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
        for cluster, pval in current_clusters:
            ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')

        ax.legend(handles=[
            *[plt.Line2D([0],[0], color=c, lw=1.5, label=l)
            for c, l in zip(['black','steelblue','cornflowerblue','tomato','salmon'],
                            ['Current'] + temporal_conds)],
            Patch(color='green',  alpha=0.4, label='Temporal > Current (p<0.05)'),
            Patch(color='orange', alpha=0.4, label='Current > Temporal (p<0.05)'),
        ], fontsize=8, ncol=4)

        ax2 = axes[1]
        ax2.plot(times, T_obs, color='purple', lw=1.5)
        ax2.fill_between(times, 0, T_obs, where=T_obs > 0, alpha=0.3, color='green')
        ax2.fill_between(times, 0, T_obs, where=T_obs < 0, alpha=0.3, color='orange')
        ax2.axhline( threshold, color='gray', ls='--', lw=0.8, label=f'threshold ±{threshold:.2f}')
        ax2.axhline(-threshold, color='gray', ls='--', lw=0.8)
        ax2.axhline(0,          color='k',   lw=0.5)
        ax2.axvline(0,          color='k',   lw=0.8, ls='--')
        ax2.set_ylabel('T statistic\n+ = Temporal > Current\n− = Current > Temporal')
        ax2.set_xlabel('Time (s)')
        ax2.legend(fontsize=8)

        for cluster, pval in temporal_clusters:
            ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
            ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                    threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkgreen')
        for cluster, pval in current_clusters:
            ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')
            ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                    -threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkorange')

        plt.tight_layout()
        plt.savefig(f"figures/{subject}/statistical_significance/{reference}/Enveloppe/Mixed_Temporal_vs_Current/{band}/cluster_perm_erp_{ch_of_interest}_{band}.png", dpi=150)
        plt.show()
        plt.close()

        print(f"\n{ch_of_interest}")
        print(f"  Temporal > Current (green):")
        for cluster, pval in temporal_clusters:
            print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
        if not temporal_clusters: print("    None")
        print(f"  Current > Temporal (orange):")
        for cluster, pval in current_clusters:
            print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
        if not current_clusters: print("    None")

stat_fun(H1): min=-2.9248827909051003 max=1.414377320548641
Running initial clustering …
Found 3 clusters


  0%|          | Permuting : 0/4999 [00:00<?,       ?it/s]


TOD2-TOD3
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None
stat_fun(H1): min=-1.919610960359764 max=2.0057199512077877
Running initial clustering …
Found 1 cluster


  0%|          | Permuting : 0/4999 [00:00<?,       ?it/s]


TOD3-TOD4
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None
stat_fun(H1): min=-2.643238924667158 max=1.7005937190719482
Running initial clustering …
Found 1 cluster


...


HAD2-HAD3
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None
stat_fun(H1): min=-3.6154718340856444 max=2.498022734960808
Running initial clustering …
Found 14 clusters


  0%|          | Permuting : 0/4999 [00:00<?,       ?it/s]


HAD3-HAD4
  Temporal > Current (green):
    None
  Current > Temporal (orange):
    None
stat_fun(H1): min=-4.001482358500111 max=2.1474525579782946
Running initial clustering …
Found 34 clusters


  0%|          | Permuting : 0/4999 [00:00<?,       ?it/s]

FileNotFoundError: [Errno 2] No such file or directory: 'figures/02/statistical_significance/bipolar/Enveloppe/Mixed_Temporal_vs_Current/high_gamma/cluster_perm_erp_TOD2-TOD3_high_gamma.png'

In [ ]:
temporal_conds = ['Past', 'Distant Past']
ch_list = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]
for band in bands_order:
    for ch_of_interest in ch_list:
        X_temporal = np.concatenate([
            epochs_bands_stats[(band, c)].get_data(picks=[ch_of_interest])[:, 0, :]
            for c in temporal_conds
        ], axis=0)
        X_current = epochs_bands_stats[(band,'Current')].get_data(picks=[ch_of_interest])[:, 0, :]


        df        = X_temporal.shape[0] + X_current.shape[0] - 2
        threshold = t_dist.ppf(1 - 0.05 / 2, df=df)

        # ── Proper signed 2-sample test ──────────────────────────────────────────
        T_obs, clusters, cluster_pv, H0 = permutation_cluster_test(
            [X_temporal, X_current],
            stat_fun=signed_ttest,
            n_permutations=5000,    # increased for more precise p-values
            threshold=threshold,
            tail=0,
            n_jobs=-1,
            seed=42,
        )
        # ─────────────────────────────────────────────────────────────────────────

        # Split clusters by sign
        temporal_clusters = []
        current_clusters  = []
        for cluster, pval in zip(clusters, cluster_pv):
            if pval < 0.05:
                if T_obs[cluster[0]].mean() > 0:
                    temporal_clusters.append((cluster, pval))
                else:
                    current_clusters.append((cluster, pval))

    
        times = epochs_bands_stats[(band, 'Current')].times
        fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

        ax = axes[0]
        for cond, color in zip(['Current'] + temporal_conds,
                            ['black', 'steelblue', 'cornflowerblue', 'tomato', 'salmon']):
            data = epochs_bands_stats[(band, cond)].get_data(picks=[ch_of_interest])[:, 0, :]
            m    = data.mean(axis=0)
            s    = data.std(axis=0) / np.sqrt(len(data))
            ax.fill_between(times, m - s, m + s, alpha=0.12, color=color)
            ax.plot(times, m, lw=1.5, label=cond, color=color)
        ax.axvline(0, color='k', lw=0.8, ls='--')
        ax.axhline(0, color='k', lw=0.4, ls=':')
        ax.set_ylabel('Amplitude (µV)')
        ax.set_title(f'Envelope: {ch_of_interest}, {band}')

        for cluster, pval in temporal_clusters:
            ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
        for cluster, pval in current_clusters:
            ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')

        ax.legend(handles=[
            *[plt.Line2D([0],[0], color=c, lw=1.5, label=l)
            for c, l in zip(['black','steelblue','cornflowerblue','tomato','salmon'],
                            ['Current'] + temporal_conds)],
            Patch(color='green',  alpha=0.4, label='Temporal > Current (p<0.05)'),
            Patch(color='orange', alpha=0.4, label='Current > Temporal (p<0.05)'),
        ], fontsize=8, ncol=4)

        ax2 = axes[1]
        ax2.plot(times, T_obs, color='purple', lw=1.5)
        ax2.fill_between(times, 0, T_obs, where=T_obs > 0, alpha=0.3, color='green')
        ax2.fill_between(times, 0, T_obs, where=T_obs < 0, alpha=0.3, color='orange')
        ax2.axhline( threshold, color='gray', ls='--', lw=0.8, label=f'threshold ±{threshold:.2f}')
        ax2.axhline(-threshold, color='gray', ls='--', lw=0.8)
        ax2.axhline(0,          color='k',   lw=0.5)
        ax2.axvline(0,          color='k',   lw=0.8, ls='--')
        ax2.set_ylabel('T statistic\n+ = Temporal > Current\n− = Current > Temporal')
        ax2.set_xlabel('Time (s)')
        ax2.legend(fontsize=8)

        for cluster, pval in temporal_clusters:
            ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
            ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                    threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkgreen')
        for cluster, pval in current_clusters:
            ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')
            ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                    -threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkorange')

        plt.tight_layout()
        plt.savefig(f"figures/{subject}/statistical_significance/{reference}/Enveloppe/All_Past_vs_Current/{band}/cluster_perm_erp_{ch_of_interest}_{band}.png", dpi=150)
        plt.show()
        plt.close()

        print(f"\n{ch_of_interest}")
        print(f"  Temporal > Current (green):")
        for cluster, pval in temporal_clusters:
            print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
        if not temporal_clusters: print("    None")
        print(f"  Current > Temporal (orange):")
        for cluster, pval in current_clusters:
            print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
        if not current_clusters: print("    None")

NameError: name 'bands_order' is not defined

In [ ]:
temporal_conds = ['Futur', 'Distant Futur']
ch_list = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]
for band in bands_order:
    for ch_of_interest in ch_list:
        X_temporal = np.concatenate([
            epochs_bands_stats[(band, c)].get_data(picks=[ch_of_interest])[:, 0, :]
            for c in temporal_conds
        ], axis=0)
        X_current = epochs_bands_stats[(band,'Current')].get_data(picks=[ch_of_interest])[:, 0, :]


        df        = X_temporal.shape[0] + X_current.shape[0] - 2
        threshold = t_dist.ppf(1 - 0.05 / 2, df=df)

        # ── Proper signed 2-sample test ──────────────────────────────────────────
        T_obs, clusters, cluster_pv, H0 = permutation_cluster_test(
            [X_temporal, X_current],
            stat_fun=signed_ttest,
            n_permutations=5000,    # increased for more precise p-values
            threshold=threshold,
            tail=0,
            n_jobs=-1,
            seed=42,
        )
        # ─────────────────────────────────────────────────────────────────────────

        # Split clusters by sign
        temporal_clusters = []
        current_clusters  = []
        for cluster, pval in zip(clusters, cluster_pv):
            if pval < 0.05:
                if T_obs[cluster[0]].mean() > 0:
                    temporal_clusters.append((cluster, pval))
                else:
                    current_clusters.append((cluster, pval))

    
        times = epochs_bands_stats[(band, 'Current')].times
        fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

        ax = axes[0]
        for cond, color in zip(['Current'] + temporal_conds,
                            ['black', 'steelblue', 'cornflowerblue', 'tomato', 'salmon']):
            data = epochs_bands_stats[(band, cond)].get_data(picks=[ch_of_interest])[:, 0, :]
            m    = data.mean(axis=0)
            s    = data.std(axis=0) / np.sqrt(len(data))
            ax.fill_between(times, m - s, m + s, alpha=0.12, color=color)
            ax.plot(times, m, lw=1.5, label=cond, color=color)
        ax.axvline(0, color='k', lw=0.8, ls='--')
        ax.axhline(0, color='k', lw=0.4, ls=':')
        ax.set_ylabel('Amplitude (µV)')
        ax.set_title(f'Envelope: {ch_of_interest}, {band}')

        for cluster, pval in temporal_clusters:
            ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
        for cluster, pval in current_clusters:
            ax.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')

        ax.legend(handles=[
            *[plt.Line2D([0],[0], color=c, lw=1.5, label=l)
            for c, l in zip(['black','steelblue','cornflowerblue','tomato','salmon'],
                            ['Current'] + temporal_conds)],
            Patch(color='green',  alpha=0.4, label='Temporal > Current (p<0.05)'),
            Patch(color='orange', alpha=0.4, label='Current > Temporal (p<0.05)'),
        ], fontsize=8, ncol=4)

        ax2 = axes[1]
        ax2.plot(times, T_obs, color='purple', lw=1.5)
        ax2.fill_between(times, 0, T_obs, where=T_obs > 0, alpha=0.3, color='green')
        ax2.fill_between(times, 0, T_obs, where=T_obs < 0, alpha=0.3, color='orange')
        ax2.axhline( threshold, color='gray', ls='--', lw=0.8, label=f'threshold ±{threshold:.2f}')
        ax2.axhline(-threshold, color='gray', ls='--', lw=0.8)
        ax2.axhline(0,          color='k',   lw=0.5)
        ax2.axvline(0,          color='k',   lw=0.8, ls='--')
        ax2.set_ylabel('T statistic\n+ = Temporal > Current\n− = Current > Temporal')
        ax2.set_xlabel('Time (s)')
        ax2.legend(fontsize=8)

        for cluster, pval in temporal_clusters:
            ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='green')
            ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                    threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkgreen')
        for cluster, pval in current_clusters:
            ax2.axvspan(times[cluster[0][0]], times[cluster[0][-1]], alpha=0.25, color='orange')
            ax2.text((times[cluster[0][0]] + times[cluster[0][-1]]) / 2,
                    -threshold * 1.1, f'p={pval:.3f}', ha='center', fontsize=8, color='darkorange')

        plt.tight_layout()
        plt.savefig(f"figures/{subject}/statistical_significance/{reference}/Enveloppe/All_Future_vs_Current/{band}/cluster_perm_erp_{ch_of_interest}_{band}.png", dpi=150)
        plt.show()
        plt.close()

        print(f"\n{ch_of_interest}")
        print(f"  Temporal > Current (green):")
        for cluster, pval in temporal_clusters:
            print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
        if not temporal_clusters: print("    None")
        print(f"  Current > Temporal (orange):")
        for cluster, pval in current_clusters:
            print(f"    {times[cluster[0][0]]:.3f} – {times[cluster[0][-1]]:.3f}s  p={pval:.4f}")
        if not current_clusters: print("    None")

In [ ]:
# Stop here
raise NotImplementedError("Stop here for now")

NotImplementedError: Stop here for now

#### Frequency bands and significant windows:

In [ ]:
temporal_conds = ['Past', 'Futur', 'Distant Past', 'Distant Futur']
#ch = ch_of_interest
ch_list = ["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]

for ch in ch_list:
    fig, axes = plt.subplots(len(bands), 1, figsize=(14, 3 * len(bands)), sharex=True)

    for ax, band in zip(axes, bands):
        ch_idx = epochs_bands_stats[(band, 'Current')].ch_names.index(ch)
        times  = epochs_bands_stats[(band, 'Current')].times

        X_temp = np.concatenate([
            epochs_bands_stats[(band, c)].get_data(picks='seeg')[:, ch_idx, :]
            for c in temporal_conds
        ], axis=0)
        X_curr = epochs_bands_stats[(band, 'Current')].get_data(picks='seeg')[:, ch_idx, :]

        # t-test at every time point
        t_vals = np.zeros(len(times))
        p_vals = np.zeros(len(times))
        for ti in range(len(times)):
            t_vals[ti], p_vals[ti] = ttest_ind(X_temp[:, ti], X_curr[:, ti])

        # Plot t-statistic
        ax.plot(times, t_vals, color='purple', lw=1.2)
        ax.axhline(0,  color='k', lw=0.5)
        ax.axvline(0,  color='k', lw=0.8, ls='--')

        # Shade uncorrected p<0.05 windows
        sig_mask = p_vals < 0.05
        ax.fill_between(times, t_vals.min(), t_vals.max(),
                        where=sig_mask, alpha=0.2, color='green',
                        label='p<0.05 uncorr.')

        ax.set_ylabel(f'{band}\nt-value', fontsize=9)
        ax.legend(fontsize=8, loc='upper right')

    axes[-1].set_xlabel('Time (s)')
    plt.suptitle(f'Time-resolved t-test: Temporal vs Current\n{ch}', y=1.01)
    plt.tight_layout()
    plt.savefig(f"figures/{subject}/statistical_significance/{reference}/Other/time_resolved_ttest_{ch}.png", dpi=150)
    plt.show()
    plt.close()

NameError: name 'epochs_bands_stats' is not defined

In [ ]:
freqs = np.concatenate([
    np.arange(4,   8,  1),    # theta
    np.arange(8,  12,  1),    # alpha
    np.arange(12, 30,  2),    # beta
    np.arange(30, 80,  5),    # low gamma
    np.arange(80, 150, 10),   # high gamma
])
n_cycles = freqs / 4.0        # shorter windows for high freqs

tfr = {}
for cond in cond_list_stats:
    tfr[cond] = mne.time_frequency.tfr_morlet(
        epochs_ref[cond],
        freqs=freqs,
        n_cycles=n_cycles,
        return_itc=False,
        average=True,
        verbose=False,
    )
    # Baseline normalize (dB)
    tfr[cond].apply_baseline(baseline=(-1.0, 0.0), mode='logratio')

# Plot difference: Temporal − Current
ch = 'HPD3-HPD4'
tfr_diff = tfr['Distant Past'].copy()
tfr_diff.data = tfr['Distant Past'].data - tfr['Current'].data


ch_idx = tfr_diff.ch_names.index(ch)
data   = tfr_diff.data[ch_idx]          # (n_freqs, n_times)
times  = tfr_diff.times
freqs  = tfr_diff.freqs

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(
    data,
    aspect='auto',
    origin='lower',
    extent=[times[0], times[-1], freqs[0], freqs[-1]],
    cmap='RdBu_r',
    vmin=-0.5, vmax=0.5,
)
ax.axvline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title(f'Distant Past − Current TFR: {ch}')
plt.colorbar(im, ax=ax, label='Log power difference')
plt.tight_layout()
plt.savefig(
    f"figures/{subject}/statistical_significance/{reference}/tfr/tfr_diff_{ch}.png",
    dpi=150, bbox_inches='tight'
)
plt.show()
plt.close()

NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Applying baseline correction (mode: logratio)
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Applying baseline correction (mode: logratio)
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Applying baseline correction (mode: logratio)
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Applying baseline correction (mode: logratio)
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Applying baseline correction (mode: logratio)
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Applying baseline correction (mode: logratio)
NOTE: tfr_morlet() is a legacy function. New code should use .compute_tfr(method="morlet").
Applying baseline correction (mode: logratio)


## <span style="color:aquamarine">Electrode placement visualisation</span>

Visualize electrode placement (especially in the region that is the focus of our research):

GIF of the brain's electrodes (uncomment code if wanted otherwise only POV screenshots)

In [ ]:
# Get transform and figure
trans = mne.transforms.Transform('mri', 'head')
fig = mne.viz.plot_alignment(
    epochs.info,
    trans=trans,         
    subject=fs_subject,
    subjects_dir=subjects_dir,
    surfaces=[],
    coord_frame="mri",
)

# Get picks and brain volume
picks_hip = mne.pick_channels(epochs.info['ch_names'], include=filtered_electrodes)

brain = mne.viz.Brain(
    fs_subject,
    alpha=0.1,
    cortex="low_contrast",
    subjects_dir=str(fs_subjects_dir), 
    units="m",
    figure=fig,
)

# If you want to color the regions of interest (hippocampus and close neighbours)
brain.add_volume_labels(aseg="aparc+aseg", labels=regions_of_interest)

# GIF to see regions of interest
# frames = []
# tmp_dir = f"figures/{subject}/tmp_frames"
# os.makedirs(tmp_dir, exist_ok=True)

# n_frames = 144  
# for i, azimuth in enumerate(np.linspace(0, 360, n_frames, endpoint=False)):
#     brain.show_view(azimuth=azimuth, elevation=70, distance=0.35)
#     screenshot = brain.screenshot()
#     if screenshot is not None:
#         frames.append(screenshot)

# imageio.mimsave(
#     f"figures/{subject}/brain_rotation.gif",
#     frames,
#     fps=20,
#     loop=0,
# )

# # Cleanup
# for f in os.listdir(tmp_dir):
#     os.remove(os.path.join(tmp_dir, f))
# os.rmdir(tmp_dir)

# print("GIF saved.")

# Add electrode labels
info_hip = mne.pick_info(epochs.info, sel=picks_hip)
brain.add_sensors(info_hip, trans=trans)
for idx in picks_hip:
    ch = epochs.info['chs'][idx]
    pos = ch['loc'][:3]  # x, y, z in head coords
    brain.plotter.add_point_labels(
        points=[pos],
        labels=[epochs.info['ch_names'][idx]],
        point_size=0,
        font_size=12,
        text_color="white",
        shape=None,
        render=False,
    )

# Set and save views
distance = 0.2
focalpoint = (-0.04,-0.05,-0.001)

brain.show_view(azimuth=120, elevation=90, distance=distance, focalpoint = focalpoint)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_mydata01.png")

brain.show_view(azimuth=60, elevation=90, distance=distance, focalpoint = focalpoint)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_mydata02.png")

brain.show_view(azimuth=60, elevation=45, distance=distance, focalpoint = focalpoint)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_mydata03.png")

brain.show_view(azimuth=0, elevation=45, distance=distance, focalpoint = focalpoint)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_mydata04.png")

brain.show_view(azimuth=30, elevation=45, distance=distance, focalpoint = focalpoint)
brain.save_image(f"figures/{subject}/regions/regions_sub{subject}_mydata05.png")

Channel types::	seeg: 17


A view with name (P_0x7ff37c5b96d0_1) is already registered
 => returning previous one


    Smoothing by a factor of 0.9
Channel types::	seeg: 17


Nice! We see that the electrodes are only located in the left hemisphere (as concluded from the labels) and the electrodes labeled as being in the hippocampus region are where they are supposed to be. We can now continue with more advanced visualizations: 

## <span style="color:aquamarine">Videos of activity</span>

For the next videos, I had to add the T1.gz from the old mri folder and lh.white and rh.white from the old surf file.
Unfortunately, I think that these do not fit with the new corrected data (to be continued).

This code follows the tutorial from MNE.

In [ ]:
# Check average response length for plotting
total=0
for i in range(len(annotations)):
    total = annotations[i]["duration"] + total
average_duration = total / len(annotations)
print(average_duration)

8.688969509803925


In [ ]:
if video_rendering == True:
    # Visualize activation in the hippocampus
    fname_src = fs_subjects_dir / patient / "bem" / f"{patient}-vol-5-src.fif"

    if not fname_src.exists():
        vol_src = mne.setup_volume_source_space(
            fs_subject,
            pos=5.0,
            subjects_dir=str(fs_subjects_dir),
            verbose=True,
        )
        mne.write_source_spaces(fname_src, vol_src, overwrite=True)
    else:
        vol_src = mne.read_source_spaces(fname_src)

    diff = mne.combine_evoked([cond_distant_past, cond_distant_futur], weights=[1, -1])

    evoked = [cond_current, cond_past, cond_futur, cond_never, cond_always, cond_distant_past, cond_distant_futur, diff]  # List of evoked objects for each condition
    evoked_names = ['Current', 'Past', 'Futur', 'Never', 'Always', 'Distant Past', 'Distant Futur','Diff']

    # TODO: zoom the video on the hippocampus
    for i in range(len(evoked)):
        if i >= nb_videos: # how many videos to render
            break

        cond = evoked[i]
        name = evoked_names[i]
        #cond.pick(picks_hip)  # Keep only the channels of interest
        stc = mne.stc_near_sensors(
            cond,
            trans,
            fs_subject,
            subjects_dir=subjects_dir,
            src=vol_src,
            surface=None,
            verbose="error",
        )
        #stc = abs(stc)  # just look at magnitude
        #clim = dict(kind="value", lims=np.percentile(cond.data, [-75, -50, -10, 10, 50, 75])
        threshold = np.percentile(np.abs(cond.data), 99)
        mid = np.percentile(np.abs(cond.data), 75)
        low = np.percentile(np.abs(cond.data), 10)

        clim = dict(kind="value", pos_lims=[low, mid, threshold])
        brain = stc.plot_3d(
            src=vol_src,
            subjects_dir=subjects_dir,
            view_layout="horizontal",
            views=["axial", "coronal", "sagittal"],
            size=(800, 300),
            show_traces=0.4,
            clim=clim,
            add_data_kwargs=dict(colorbar_kwargs=dict(label_font_size=8)),
        )
        brain.save_movie(time_dilation=3, interpolation='linear', framerate=24, time_viewer=True, filename=f'videos/{subject}/{reference}/new_filtering/{patient}_ieeg_{str(name)}_{reference}_70-200filt.mp4')

## Time frequency analysis

To be ignored for now

#### Frequency analysis

In [ ]:
if reference == "bipolar" :
    epochs_ref_for_psd = mne.set_bipolar_reference(
        epochs,
        ["TOD2", "TOD3", "PHD1", "PHD2", "PHD3", "PHD4", "HPD1", "HPD2", "HPD3", "HPD4", "HAD1", "HAD2", "HAD3"],
        ["TOD3", "TOD4", "PHD2", "PHD3", "PHD4", "PHD5", "HPD2", "HPD3", "HPD4", "HPD5", "HAD2", "HAD3", "HAD4"],
        ch_name=["TOD2-TOD3", "TOD3-TOD4", "PHD1-PHD2", "PHD2-PHD3", "PHD3-PHD4", "PHD4-PHD5", "HPD1-HPD2", "HPD2-HPD3", "HPD3-HPD4", "HPD4-HPD5", "HAD1-HAD2", "HAD2-HAD3", "HAD3-HAD4"]
    )
else:
    epochs_ref_for_psd = epochs_ref

sEEG channel type selected for re-referencing
Not setting metadata
102 matching events found
No baseline correction applied
0 projection items activated
Added the following bipolar channels:
TOD2-TOD3, TOD3-TOD4, PHD1-PHD2, PHD2-PHD3, PHD3-PHD4, PHD4-PHD5, HPD1-HPD2, HPD2-HPD3, HPD3-HPD4, HPD4-HPD5, HAD1-HAD2, HAD2-HAD3, HAD3-HAD4


In [ ]:
# Get the frequency spectrum at different stages of filtering
fmin, fmax = 0.1, 400

if reference == "bipolar":
    epochs_ref_for_psd.compute_psd(fmin=fmin, fmax=fmax).plot(
        average=True, amplitude=False, picks="data", exclude="bads"
    ).savefig(f"figures/{subject}/psd/{subject}_psd_{reference}_epochs_ref_for_psd.png", dpi=300)

# After epochs where referenced
epochs_ref.compute_psd(fmin=fmin, fmax=fmax).plot(
    average=True, amplitude=False, picks="data", exclude="bads"
).savefig(f"figures/{subject}/psd/{subject}_psd_{reference}.png", dpi=300)

# Just after epochs are created
epochs.compute_psd(fmin=fmin, fmax=fmax).plot(
    average=True, amplitude=False, picks="data", exclude="bads"
).savefig(f"figures/{subject}/psd/{subject}_psd_epochs.png", dpi=300)

# After data was loaded with no pre-processing
raw_for_psd = raw.copy().pick_channels(filtered_electrodes) 
raw_for_psd.compute_psd(fmin=fmin, fmax=fmax).plot(
    average=True, amplitude=False, picks="data", exclude="bads"
).savefig(f"figures/{subject}/psd/{subject}_psd_raw.png", dpi=300)

# After the first filter (bandpass 0.1 - 250Hz)
raw_filtered.compute_psd(fmin=fmin, fmax=fmax).plot(
    average=True, amplitude=False, picks="data", exclude="bads"
).savefig(f"figures/{subject}/psd/{subject}_psd_raw_filtered.png", dpi=300)

    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).
Averaging across epochs before plotting...
    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).
Averaging across epochs before plotting...
    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).
Averaging across epochs before plotting...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Effective window size : 1.000 (s)
Plotting power spectral density (dB=True).
Effective window size : 1.000 (s)
Plotting power spectral density (dB=True).


In [ ]:
# More suited for eeg with all electrodes
epochs.compute_psd().plot_topomap(ch_type=None, normalize=False, contours=0).savefig(f"figures/{subject}/psd/{subject}_psd_topomap_{reference}.png", dpi=300)

    Using multitaper spectrum estimation with 7 DPSS windows
Averaging across epochs before plotting...


In [ ]:
_, ax = plt.subplots()
spectrum = epochs_ref.compute_psd(fmin=2.0, fmax=200.0, tmax=3.0, n_jobs=None)
# average across epochs first
mean_spectrum = spectrum.average()
psds, freqs = mean_spectrum.get_data(return_freqs=True)
# then convert to dB and take mean & standard deviation across channels
psds = 10 * np.log10(psds)
psds_mean = psds.mean(axis=0)
psds_std = psds.std(axis=0)

ax.plot(freqs, psds_mean, color="k")
ax.fill_between(
    freqs,
    psds_mean - psds_std,
    psds_mean + psds_std,
    color="k",
    alpha=0.5,
    edgecolor="none",
)
ax.set(
    title="Multitaper PSD (gradiometers)",
    xlabel="Frequency (Hz)",
    ylabel="Power Spectral Density (dB)",
)

ax.figure.savefig(f"figures/{subject}/psd/{subject}_psd_multitaper_{reference}.png", dpi=300)

    Using multitaper spectrum estimation with 7 DPSS windows


#### Time-frequency analysis

In [ ]:
freqs = np.logspace(*np.log10([6, 35]), num=8)
n_cycles = freqs / 2.0  # different number of cycle per frequency
power, itc = epochs_ref.compute_tfr(
    method="morlet",
    freqs=freqs,
    n_cycles=n_cycles,
    average=True,
    return_itc=True,
    decim=3,
)

#### Power

In [ ]:
power.plot_topo(baseline=(-0.5, 0), mode="logratio", title="Average power")
power.plot(picks=[10], baseline=(-0.5, 0), mode="logratio", title=power.ch_names[10])

fig, axes = plt.subplots(1, 2, figsize=(7, 4), layout="constrained")
topomap_kw = dict(
    ch_type=None, tmin=0.5, tmax=1.5, baseline=(-0.5, 0), mode="logratio", show=False
)
plot_dict = dict(Alpha=dict(fmin=8, fmax=12), Beta=dict(fmin=13, fmax=25))
for ax, (title, fmin_fmax) in zip(axes, plot_dict.items()):
    power.plot_topomap(**fmin_fmax, axes=ax, **topomap_kw)
    ax.set_title(title)
fig.savefig(f"figures/{subject}/psd/{subject}_tfr_topomap_{reference}.png", dpi=300)

Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)
Applying baseline correction (mode: logratio)


In [ ]:
power.plot_joint(
    baseline=(-0.5, 0), mode="mean", tmin=-0.5, tmax=2, timefreqs=[(0.5, 10), (1.3, 8)]
).savefig(f"figures/{subject}/psd/{subject}_tfr_joint_{reference}.png", dpi=300)

Applying baseline correction (mode: mean)
Applying baseline correction (mode: mean)


Applying baseline correction (mode: mean)
